In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# Multivariate Spectral Granger Causality Analysis

This notebook computes **multivariate state-space Granger causality** between raw LFP signals from electrodes in different ROIs using `mne_connectivity.spectral_connectivity_epochs`.

**Key features:**
- Uses multivariate state-space GC on raw LFP epochs (theta band: 4-8 Hz)
- Computes GC across epochs **grouped by RPE category** (neg, neutral, pos)
- Time-resolved output: GC dynamics from 0 to 1.5s post-feedback
- Multivariate approach: considers all electrodes within each ROI jointly
- Tests directionality: ACC → Target vs Target → ACC

**Analysis approach:**
- Source ROI: ACC
- Target ROIs: dmPFC, OFC, dlPFC, HPC, AMY
- Method: `spectral_connectivity_epochs` with methods 'gc' (forward) and 'gc_tr' (backward)

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import mne
from scipy.signal import filtfilt
from scipy.ndimage import uniform_filter1d
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm
from IPython.display import clear_output
import warnings

# MNE-Connectivity for multivariate spectral Granger causality
from mne_connectivity import spectral_connectivity_epochs

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Add LFPAnalysis to path
import sys
sys.path.append('/Users/salmanqasim/Documents/GitRepos/LFPAnalysis')
from LFPAnalysis import oscillation_utils

# Import plot config if available
try:
    from membandit.plotting_style import set_plot_style
    set_plot_style()
except:
    pass

print("Imports loaded successfully (including mne_connectivity for multivariate GC)")

## Setup paths and load demographics

In [ ]:
# Base directory for data

base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load the demographics data
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Filter out problematic subjects
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]

# Get subject info
subj_ids = subj_df.MSSMCode.values
print(f"Available subjects ({len(subj_ids)}): {subj_ids}")

In [ ]:
# Load behavioral data (encoding/learning phase only)

learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')

# Add log reaction times
learn_df['gamble_rt_log'] = np.log(learn_df.gamble_rt)

# Filter out fast reaction times
# learn_df = learn_df[learn_df.gamble_rt > 0.3]

# Add standardized behavioral variables per subject
for subj in learn_df.participant.unique():
    # Z-score RPE within subject (Gelman standardization)
    learn_df.loc[learn_df.participant==subj, 'zrpe'] = (
        learn_df.loc[learn_df.participant==subj, 'rpe'] -
        np.nanmean(learn_df.loc[learn_df.participant==subj, 'rpe'])
    ) / (2 * np.nanstd(learn_df.loc[learn_df.participant==subj, 'rpe']))

    # Add RPE category
    learn_df.loc[(learn_df.participant==subj), 'rpe_cat'] = pd.qcut(
        learn_df.loc[(learn_df.participant==subj), 'rpe'], 
        3, labels=['neg', 'neutral', 'pos']
    )

print(f"Loaded learn_df: {learn_df.shape}")

In [ ]:
learn_df

## Epoch Loading and Multivariate Indices Functions

In [ ]:
def get_roi_channels(elec_df, epochs, roi, hemisphere=None):
    """
    Get channel names and indices for electrodes in a specific ROI.
    """
    roi_mask = elec_df['salman_region'] == roi
    if hemisphere is not None:
        hemi_mask = elec_df['hemisphere'].str.lower() == hemisphere.lower()
        roi_mask = roi_mask & hemi_mask
    roi_channels = elec_df[roi_mask]['label'].tolist()
    all_ch_names = epochs.ch_names
    valid_channels = [ch for ch in roi_channels if ch in all_ch_names]
    indices = [all_ch_names.index(ch) for ch in valid_channels]
    return {
        'channels': valid_channels,
        'indices': np.array(indices),
        'n_channels': len(valid_channels)
    }

def make_multivariate_indices(source_indices, target_indices):
    """
    Create multivariate indices for spectral_connectivity_epochs.
    """
    return (
        [np.array(source_indices)],
        [np.array(target_indices)]
    )


def select_top_electrode_per_roi(epochs, roi_channels, fmin, fmax, verbose=False):
    """
    Select the single electrode with highest median power in a frequency band.

    Parameters
    ----------
    epochs : mne.Epochs
        MNE epochs object containing LFP data
    roi_channels : list
        List of channel names in the ROI
    fmin, fmax : float
        Frequency band of interest (Hz)
    verbose : bool
        Whether to print selection information

    Returns
    -------
    dict : Dictionary containing:
        - 'electrode': Name of selected electrode (or None if no channels)
        - 'median_power': Median power value of selected electrode
        - 'all_powers': Dictionary mapping all electrodes -> median power
    """
    if len(roi_channels) == 0:
        return {
            'electrode': None,
            'median_power': 0.0,
            'all_powers': {}
        }

    # Filter to only channels that exist in epochs
    valid_channels = [ch for ch in roi_channels if ch in epochs.ch_names]
    
    if len(valid_channels) == 0:
        return {
            'electrode': None,
            'median_power': 0.0,
            'all_powers': {}
        }

    # Compute median power for each electrode
    electrode_powers = {}
    
    for ch_name in valid_channels:
        # Pick single channel
        epochs_ch = epochs.copy().pick_channels([ch_name])
        
        # Filter to frequency band
        epochs_ch = epochs_ch.copy().filter(fmin, fmax, verbose=False)
        
        # Compute power (squared amplitude) for each trial
        power_data = epochs_ch.get_data() ** 2  # (n_epochs, 1, n_times)
        
        # Compute median power across time for each trial, then median across trials
        trial_medians = np.median(power_data[:, 0, :], axis=1)  # Median across time per trial
        median_power = np.median(trial_medians)  # Median across trials
        electrode_powers[ch_name] = median_power

    # Select electrode with highest median power
    if len(electrode_powers) > 0:
        top_electrode = max(electrode_powers, key=electrode_powers.get)
        top_power = electrode_powers[top_electrode]
        
        if verbose:
            print(f"    Selected {top_electrode} (median power: {top_power:.6f})")
        
        return {
            'electrode': top_electrode,
            'median_power': top_power,
            'all_powers': electrode_powers
        }
    else:
        return {
            'electrode': None,
            'median_power': 0.0,
            'all_powers': {}
        }

def load_subject_epochs(subj_id, base_dir, event, roi_source, roi_target,
                         tmin=None, tmax=None, target_sfreq=50, day='Day1',
                         filter_by_power=False, power_percentile=50, fmin=None, fmax=None,
                         tfr_time_window=None, use_top_electrode_only=False):
    """
    Load raw LFP epochs for a subject, downsample, and identify ROI channels.

    Parameters
    ----------
    subj_id : str
        Subject ID
    base_dir : str
        Base directory for data
    event : str
        Event name (e.g., 'feedback_start')
    roi_source : str
        Source ROI name (e.g., 'ACC')
    roi_target : str
        Target ROI name (e.g., 'OFC')
    tmin, tmax : float, optional
        Time window to crop epochs
    target_sfreq : float
        Target sampling frequency for downsampling (default: 50 Hz)
        Lower sfreq allows longer lags in GC computation.
        At 50 Hz, gc_n_lags=7 gives 140 ms of temporal history.
    day : str
        Day folder (default: 'Day1')
    filter_by_power : bool, optional
        If True, filter electrodes based on TFR band power (default: False)
    power_percentile : float, optional
        Percentile threshold for filtering (default: 50 = top 50%)
    fmin, fmax : float, optional
        Frequency band for power filtering (Hz)
    tfr_time_window : tuple, optional
        (tmin, tmax) for time averaging in TFR, None = all times
    use_top_electrode_only : bool, optional
        If True, select only the top electrode per ROI per hemisphere based on median power
        in the frequency band (default: False). Requires fmin and fmax to be specified.

    Returns
    -------
    dict : Dictionary with epochs and channel info, or None if failed
    """
    try:
        # Load epochs
        load_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
        epochs_file = f'{load_path}/{event}-epo.fif'

        if not os.path.exists(epochs_file):
            print(f"  Epochs file not found: {epochs_file}")
            return None

        epochs = mne.read_epochs(epochs_file, preload=True, verbose=False)
        original_sfreq = epochs.info['sfreq']

        # Crop to time window if specified
        if tmin is not None or tmax is not None:
            epochs = epochs.crop(tmin=tmin, tmax=tmax)

        # Downsample to target sampling frequency
        if target_sfreq < original_sfreq:
            print(f"  Downsampling: {original_sfreq} Hz -> {target_sfreq} Hz")
            epochs = epochs.resample(target_sfreq, verbose=False)

        actual_sfreq = epochs.info['sfreq']

        # Load electrode dataframe
        elec_df_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df'
        elec_df = pd.read_csv(elec_df_path)

        # Get ROI channels for each hemisphere
        result = {
            'epochs': epochs,
            'times': epochs.times,
            'sfreq': actual_sfreq,
            'original_sfreq': original_sfreq,
            'n_epochs': len(epochs)
        }

        # Check if we need to select top electrodes
        if use_top_electrode_only:
            if fmin is None or fmax is None:
                print(f"  Warning: use_top_electrode_only=True requires fmin and fmax. Using all electrodes.")
                use_top_electrode_only = False
            else:
                print(f"  Selecting top electrode per ROI per hemisphere (based on median power in {fmin}-{fmax} Hz)")

        for hemi in ['l', 'r']:
            # Source ROI channels
            source_info = get_roi_channels(elec_df, epochs, roi_source, hemisphere=hemi)
            source_channels = source_info['channels']
            
            # Select top electrode if requested
            if use_top_electrode_only and len(source_channels) > 0:
                top_source = select_top_electrode_per_roi(epochs, source_channels, fmin, fmax, verbose=True)
                if top_source['electrode'] is not None:
                    source_channels = [top_source['electrode']]
                    print(f"  Hemisphere {hemi} - Source {roi_source}: {top_source['electrode']}")
                else:
                    source_channels = []
            
            result[f'source_channels_{hemi}'] = source_channels
            result[f'source_indices_{hemi}'] = np.array([epochs.ch_names.index(ch) for ch in source_channels if ch in epochs.ch_names])
            result[f'n_source_{hemi}'] = len(source_channels)

            # Target ROI channels
            target_info = get_roi_channels(elec_df, epochs, roi_target, hemisphere=hemi)
            target_channels = target_info['channels']
            
            # Select top electrode if requested
            if use_top_electrode_only and len(target_channels) > 0:
                top_target = select_top_electrode_per_roi(epochs, target_channels, fmin, fmax, verbose=True)
                if top_target['electrode'] is not None:
                    target_channels = [top_target['electrode']]
                    print(f"  Hemisphere {hemi} - Target {roi_target}: {top_target['electrode']}")
                else:
                    target_channels = []
            
            result[f'target_channels_{hemi}'] = target_channels
            result[f'target_indices_{hemi}'] = np.array([epochs.ch_names.index(ch) for ch in target_channels if ch in epochs.ch_names])
            result[f'n_target_{hemi}'] = len(target_channels)

        return result

    except Exception as e:
        print(f"  Error loading epochs for {subj_id}: {e}")
        return None

## Multivariate Spectral Granger Causality Functions

In [ ]:
def compute_multivariate_gc(epochs, source_indices, target_indices,
                            fmin=4, fmax=8, n_cycles=5,
                            gc_n_lags=7, n_freqs=None, verbose=False):
    """
    Compute multivariate state-space Granger causality using mne_connectivity.
    Uses spectral_connectivity_epochs with cwt_morlet mode and methods 'gc' (forward)
    and 'gc_tr' (backward) to compute time-resolved, frequency-averaged GC between ROIs.

    Parameters
    ----------
    epochs : mne.Epochs
        MNE epochs object containing LFP data
    source_indices : array-like
        Indices of source ROI channels in epochs
    target_indices : array-like
        Indices of target ROI channels in epochs
    fmin, fmax : float
        Frequency band (default: theta 4-8 Hz)
    n_cycles : int or float
        Number of cycles for wavelet (default: 5)
    gc_n_lags : int
        Number of lags for state-space model
    verbose : bool
        Whether to print progress

    Returns
    -------
    dict : Dictionary containing:
        - gc_forward: array (n_times,) - Forward GC (source -> target), averaged across freqs
        - gc_backward: array (n_times,) - Backward GC (target -> source), averaged across freqs
        - gc_net: array (n_times,) - Net GC (forward - backward)
        - times: array - Time points
        - freqs: array - Frequency points used
        - success: bool - Whether computation succeeded
    """
    result = {
        'gc_forward': None,
        'gc_backward': None,
        'gc_net': None,
        'times': None,
        'freqs': None,
        'success': False
    }

    # Validate inputs
    if len(source_indices) == 0 or len(target_indices) == 0:
        if verbose:
            print("  No channels in source or target ROI")
        return result

    if len(epochs) < 5:
        if verbose:
            print(f"  Insufficient epochs: {len(epochs)}")
        return result

    try:
        # Define frequencies for cwt_morlet (within theta band)
        # Determine number of frequencies (use n_freqs if provided, else auto)
        if n_freqs is None:
            # Auto-determine based on bandwidth
            bandwidth = fmax - fmin
            if bandwidth < 10:
                n_freqs = 20
            elif bandwidth < 20:
                n_freqs = 10
            else:
                n_freqs = 5

        cwt_freqs = np.linspace(fmin, fmax, n_freqs)

        # Compute forward GC (source -> target)
        if verbose:
            print(f"    Computing forward GC ({len(source_indices)} -> {len(target_indices)} channels)...")

        # Create indices for forward direction: source -> target
        indices_fwd = make_multivariate_indices(source_indices, target_indices)

        

        con_fwd = spectral_connectivity_epochs(
            epochs,
            method='gc',
            indices=indices_fwd,
            mode='cwt_morlet',
            cwt_freqs=cwt_freqs,
            cwt_n_cycles=n_cycles,
            gc_n_lags=gc_n_lags,
            verbose=False
        )

        # Compute backward GC (target -> source) - SWAP indices!
        if verbose:
            print(f"    Computing backward GC ({len(target_indices)} -> {len(source_indices)} channels)...")

        # Create indices with swapped order: target -> source
        indices_bwd = make_multivariate_indices(target_indices, source_indices)

        con_bwd = spectral_connectivity_epochs(
            epochs,
            method='gc',  # Use regular GC with swapped indices
            indices=indices_bwd,
            mode='cwt_morlet',
            cwt_freqs=cwt_freqs,
            cwt_n_cycles=n_cycles,
            gc_n_lags=gc_n_lags,
            verbose=False
        )

        # Extract GC values - shape is (n_connections, n_freqs, n_times)
        gc_fwd = con_fwd.get_data()
        gc_bwd = con_bwd.get_data()

        if verbose:
            print(f"    Raw shape: {gc_fwd.shape}")

        # Squeeze out connection dimension (we only have 1 connection)
        gc_fwd = np.squeeze(gc_fwd)  # Now (n_freqs, n_times) or (n_times,)
        gc_bwd = np.squeeze(gc_bwd)

        # Average across frequencies to get time-resolved output
        if gc_fwd.ndim > 1:
            gc_fwd = np.mean(gc_fwd, axis=0)  # Average across freqs -> (n_times,)
            gc_bwd = np.mean(gc_bwd, axis=0)

        result['gc_forward'] = gc_fwd
        result['gc_backward'] = gc_bwd
        result['gc_net'] = gc_fwd - gc_bwd
        result['times'] = con_fwd.times
        result['freqs'] = cwt_freqs
        result['success'] = True

        if verbose:
            print(f"    Success! Mean forward GC: {np.mean(gc_fwd):.4f}, backward: {np.mean(gc_bwd):.4f}")

    except Exception as e:
        if verbose:
            print(f"    GC computation failed: {e}")
        result['error'] = str(e)

    return result





def compute_gc_by_rpe_category(epoch_data, learn_df, subj_id,
                               fmin=4, fmax=8, gc_n_lags=7, n_freqs=None, verbose=False):
    """
    Compute GC separately for each RPE category (neg, neutral, pos).

    Parameters
    ----------
    epoch_data : dict
        Output from load_subject_epochs()
    learn_df : pd.DataFrame
        Behavioral dataframe with RPE categories
    subj_id : str
        Subject ID
    fmin, fmax : float
        Frequency band
    gc_n_lags : int
        State-space model order
    verbose : bool
        Print progress

    Returns
    -------
    dict : Results for each hemisphere with GC per RPE category
    """
    epochs = epoch_data['epochs']

    # Get behavioral data for this subject
    subj_behavior = learn_df[learn_df.participant == subj_id].copy()

    # Ensure trial numbers align
    n_epochs = len(epochs)
    if len(subj_behavior) != n_epochs:
        if verbose:
            print(f"  Warning: {len(subj_behavior)} behavior rows vs {n_epochs} epochs")
        subj_behavior = subj_behavior.iloc[:n_epochs]

    results = {}

    for hemi in ['l', 'r']:
        source_idx = epoch_data.get(f'source_indices_{hemi}', np.array([]))
        target_idx = epoch_data.get(f'target_indices_{hemi}', np.array([]))
        n_source = epoch_data.get(f'n_source_{hemi}', 0)
        n_target = epoch_data.get(f'n_target_{hemi}', 0)

        if n_source == 0 or n_target == 0:
            if verbose:
                print(f"  Hemisphere {hemi}: No electrodes in source or target")
            continue

        if verbose:
            print(f"  Hemisphere {hemi}: {n_source} source, {n_target} target channels")

        hemi_results = {
            'n_source': n_source,
            'n_target': n_target,
            'source_channels': epoch_data.get(f'source_channels_{hemi}', []),
            'target_channels': epoch_data.get(f'target_channels_{hemi}', [])
        }

        # Process each RPE category
        for rpe_cat in ['neg', 'neutral', 'pos']:
            cat_mask = subj_behavior['rpe_cat'] == rpe_cat
            cat_trials = np.where(cat_mask.values)[0]

            if len(cat_trials) < 5:
                if verbose:
                    print(f"    {rpe_cat}: Only {len(cat_trials)} trials, skipping")
                hemi_results[rpe_cat] = {'success': False, 'n_trials': len(cat_trials)}
                continue

            if verbose:
                print(f"    {rpe_cat}: {len(cat_trials)} trials")

            epochs_cat = epochs[cat_trials]

            gc_result = compute_multivariate_gc(
                epochs_cat, source_idx, target_idx,
                fmin=fmin, fmax=fmax, gc_n_lags=gc_n_lags, n_freqs=n_freqs,
                verbose=verbose
            )
            gc_result['n_trials'] = len(cat_trials)
            hemi_results[rpe_cat] = gc_result

        results[hemi] = hemi_results

    return results





def combine_hemispheres(gc_by_hemi):
    """
    Combine GC results across hemispheres using simple averaging.

    Parameters
    ----------
    gc_by_hemi : dict
        Dictionary with keys 'l' and/or 'r' containing GC results per hemisphere

    Returns
    -------
    dict : Combined results with GC per RPE category (hemispheres averaged)
    """
    rpe_cats = ['neg', 'neutral', 'pos']
    combined = {}

    for rpe_cat in rpe_cats:
        # Collect valid results from each hemisphere
        valid_results = []
        total_trials = 0
        times = None

        for hemi in ['l', 'r']:
            if hemi not in gc_by_hemi:
                continue
            hemi_data = gc_by_hemi[hemi]
            if rpe_cat not in hemi_data:
                continue
            cat_data = hemi_data[rpe_cat]
            if not cat_data.get('success', False):
                continue

            valid_results.append(cat_data)
            total_trials += cat_data.get('n_trials', 0)
            if times is None:
                times = cat_data.get('times', None)

        if len(valid_results) == 0:
            combined[rpe_cat] = {'success': False, 'n_trials': 0}
            continue

        # Simple average across hemispheres (no weighting)
        gc_forward = np.mean([r['gc_forward'] for r in valid_results], axis=0)
        gc_backward = np.mean([r['gc_backward'] for r in valid_results], axis=0)
        gc_net = np.mean([r['gc_net'] for r in valid_results], axis=0)

        combined[rpe_cat] = {
            'success': True,
            'gc_forward': gc_forward,
            'gc_backward': gc_backward,
            'gc_net': gc_net,
            'times': times,
            'n_trials': total_trials,
            'n_hemispheres': len(valid_results)
        }

    return combined





print("Multivariate spectral GC functions defined (using spectral_connectivity_epochs with cwt_morlet)")

In [ ]:
def process_subject_gc(subj_id, roi_source, roi_target, base_dir, learn_df,
                       event='feedback_start', tmin=0.0, tmax=1.5,
                       target_sfreq=50, fmin=4, fmax=8, gc_n_lags=10, n_freqs=None, verbose=False,
                       filter_by_power=False, power_percentile=50, tfr_time_window=None,
                       use_top_electrode_only=False):

    """

    Process a single subject: load epochs (with downsampling) and compute GC by RPE category.

    Parameters
    ----------
    subj_id : str
        Subject ID
    roi_source : str
        Source ROI (e.g., 'ACC')
    roi_target : str
        Target ROI (e.g., 'OFC')
    base_dir : str
        Base directory path
    learn_df : pd.DataFrame
        Behavioral data with RPE categories
    event : str
        Event name
    tmin, tmax : float
        Time window
    target_sfreq : float
        Target sampling frequency for downsampling (Hz)
    fmin, fmax : float
        Frequency band (Hz)
    gc_n_lags : int
        State-space model order
    n_freqs : int, optional
        Number of frequency bins
    verbose : bool
        Print progress
    filter_by_power : bool
        If True, filter electrodes based on TFR band power
    power_percentile : float
        Percentile threshold for filtering (default: 50 = top 50%)
    tfr_time_window : tuple, optional
        (tmin, tmax) for time averaging in TFR
    use_top_electrode_only : bool, optional
        If True, select only the top electrode per ROI per hemisphere based on median power
        in the frequency band (default: False). Requires fmin and fmax to be specified.

    Returns
    -------
    dict : Results dictionary or None if failed
    """
    if verbose:
        print(f"\nProcessing {subj_id}: {roi_source} -> {roi_target}")

    # Load epochs with downsampling
    epoch_data = load_subject_epochs(
        subj_id, base_dir, event, roi_source, roi_target,
        tmin=tmin, tmax=tmax, target_sfreq=target_sfreq,
        filter_by_power=filter_by_power, power_percentile=power_percentile,
        fmin=fmin, fmax=fmax, tfr_time_window=tfr_time_window,
        use_top_electrode_only=use_top_electrode_only
    )

    if epoch_data is None:
        if verbose:
            print(f"  Failed to load epochs")
        return None

    # Compute GC by RPE category
    gc_results = compute_gc_by_rpe_category(
        epoch_data, learn_df, subj_id,
        fmin=fmin, fmax=fmax, gc_n_lags=gc_n_lags, n_freqs=n_freqs,
        verbose=verbose
    )

    if len(gc_results) == 0:
        if verbose:
            print(f"  No valid results")
        return None

    # Package results
    result = {
        'participant': subj_id,
        'roi_source': roi_source,
        'roi_target': roi_target,
        'event': event,
        'times': epoch_data['times'],
        'sfreq': epoch_data['sfreq'],
        'original_sfreq': epoch_data.get('original_sfreq', epoch_data['sfreq']),
        'n_epochs_total': epoch_data['n_epochs'],
        'gc_by_hemi': gc_results
    }

    return result

print("Subject processing function defined (with downsampling)")

In [ ]:
def export_gc_to_dataframe(result_dict, subj_id, roi_source, roi_target):

    """
    Convert GC results to long-format DataFrame with time-resolved values.
    Parameters
    ----------
    result_dict : dict
        Output from process_subject_gc() with combined hemispheres
    subj_id : str
        Subject ID
    roi_source : str
        Source ROI
    roi_target : str
        Target ROI

    Returns
    -------
    pd.DataFrame : DataFrame with columns: participant, roi_source, roi_target, 
                   rpe_cat, time, gc_forward, gc_backward, gc_net, n_trials
    """

    # Combine hemispheres
    combined_gc = combine_hemispheres(result_dict['gc_by_hemi'])

    rows = []
    rpe_cats = ['neg', 'neutral', 'pos']

    for rpe_cat in rpe_cats:
        if rpe_cat not in combined_gc or not combined_gc[rpe_cat].get('success', False):
            continue

        gc_data = combined_gc[rpe_cat]
        times = gc_data['times']
        n_trials = gc_data['n_trials']

        for t_idx, t in enumerate(times):
            rows.append({
                'participant': subj_id,
                'roi_source': roi_source,
                'roi_target': roi_target,
                'rpe_cat': rpe_cat,
                'time': t,
                'gc_forward': gc_data['gc_forward'][t_idx],
                'gc_backward': gc_data['gc_backward'][t_idx],
                'gc_net': gc_data['gc_net'][t_idx],
                'n_trials': n_trials
            })

    return pd.DataFrame(rows)


def compute_band_power_by_rpe(epoch_data, learn_df, subj_id, fmin=4, fmax=8):
    """
    Compute frequency band power for source and target ROIs, grouped by RPE category.

    Parameters
    ----------
    epoch_data : dict
        Output from load_subject_epochs()
    learn_df : pd.DataFrame
        Behavioral dataframe with RPE categories
    subj_id : str
        Subject ID
    fmin, fmax : float
        Frequency band limits (Hz)

    Returns
    -------
    dict : Dictionary with band power results for source and target ROIs per RPE category
        Structure: {
            'source': {'neg': {...}, 'neutral': {...}, 'pos': {...}},
            'target': {'neg': {...}, 'neutral': {...}, 'pos': {...}},
            'times': array
        }
        Each RPE category dict contains: 'mean', 'sem', 'n_trials'
    """
    epochs = epoch_data['epochs']
    times = epoch_data['times']

    # Get behavioral data for this subject
    subj_behavior = learn_df[learn_df.participant == subj_id].copy()

    # Ensure trial numbers align
    n_epochs = len(epochs)
    if len(subj_behavior) != n_epochs:
        subj_behavior = subj_behavior.iloc[:n_epochs]

    # Initialize results structure
    results = {
        'source': {'neg': None, 'neutral': None, 'pos': None},
        'target': {'neg': None, 'neutral': None, 'pos': None},
        'times': times
    }

    # Process each ROI type
    for roi_type in ['source', 'target']:
        # Get channels for this ROI (combine hemispheres)
        all_channels = []
        for hemi in ['l', 'r']:
            channels = epoch_data.get(f'{roi_type}_channels_{hemi}', [])
            all_channels.extend(channels)

        if len(all_channels) == 0:
            continue

        # Pick only ROI channels
        epochs_roi = epochs.copy().pick_channels(all_channels)

        # Filter to frequency band
        epochs_roi = epochs_roi.copy().filter(fmin, fmax, verbose=False)

        # Compute power (squared amplitude)
        power_data = epochs_roi.get_data() ** 2  # (n_epochs, n_channels, n_times)

        # Average across channels within ROI
        power_roi = np.mean(power_data, axis=1)  # (n_epochs, n_times)

        # Process each RPE category
        for rpe_cat in ['neg', 'neutral', 'pos']:
            cat_mask = subj_behavior['rpe_cat'] == rpe_cat
            cat_trials = np.where(cat_mask.values)[0]

            if len(cat_trials) < 1:
                continue

            # Get power for this category
            cat_power = power_roi[cat_trials, :]  # (n_trials, n_times)

            # Compute mean and SEM across trials
            mean_power = np.mean(cat_power, axis=0)
            sem_power = np.std(cat_power, axis=0) / np.sqrt(len(cat_trials))

            results[roi_type][rpe_cat] = {
                'mean': mean_power,
                'sem': sem_power,
                'n_trials': len(cat_trials)
            }

    return results


def generate_subject_plot(subj_id, roi_source, roi_target, result_dict, epoch_data, 

                          learn_df, fmin=4, fmax=8, band_name=None, savepath=None):

    """

    Generate 2x3 visualization: frequency band power (row 1) + GC time series (row 2).

    Parameters
    ----------
    subj_id : str
        Subject ID
    roi_source : str
        Source ROI
    roi_target : str
        Target ROI
    result_dict : dict
        Output from process_subject_gc()
    epoch_data : dict
        Output from load_subject_epochs()
    learn_df : pd.DataFrame
        Behavioral data
    fmin, fmax : float
        Frequency band
    band_name : str, optional
        Band name (e.g., 'theta', 'alpha'). If None, will be determined from fmin/fmax
    savepath : str, optional
        If provided, save plot to this path

    Returns
    -------
    matplotlib.figure.Figure : The figure object
    """

    # Combine GC results across hemispheres

    combined_gc = combine_hemispheres(result_dict['gc_by_hemi'])

    # Compute frequency band power for source and target ROIs
    band_power_results = compute_band_power_by_rpe(
        epoch_data, learn_df, subj_id,
        fmin=fmin, fmax=fmax
    )

    # Check which RPE categories have valid data
    rpe_cats = ['neg', 'neutral', 'pos']
    valid_cats = [cat for cat in rpe_cats if cat in combined_gc and combined_gc[cat].get('success', False)]

    if len(valid_cats) < 1:
        print(f"  No valid RPE categories for plotting")
        return None

    # Get times from GC (this is the reference)
    times = combined_gc[valid_cats[0]]['times']

    # Verify and fix time alignment - ensure band power matches GC times
    band_power_times = band_power_results.get('times', None)
    if band_power_times is not None and len(band_power_times) != len(times):
        from scipy.interpolate import interp1d
        print(f"  Interpolating band power: {len(band_power_times)} -> {len(times)} timepoints")

        for roi_type in ['source', 'target']:
            for cat in rpe_cats:
                if band_power_results[roi_type].get(cat) is not None:
                    old_mean = band_power_results[roi_type][cat]['mean']
                    old_sem = band_power_results[roi_type][cat]['sem']

                    # Only interpolate if lengths don't match
                    if len(old_mean) != len(times):
                        # Interpolate mean and sem
                        interp_mean = interp1d(band_power_times, old_mean, kind='linear', 
                                              bounds_error=False, fill_value='extrapolate')
                        interp_sem = interp1d(band_power_times, old_sem, kind='linear',
                                             bounds_error=False, fill_value='extrapolate')

                        band_power_results[roi_type][cat]['mean'] = interp_mean(times)
                        band_power_results[roi_type][cat]['sem'] = interp_sem(times)

        # Update times in band_power_results to match
        band_power_results['times'] = times

    # Create 2x3 figure
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))

    # Colors and labels for RPE categories
    colors = {'neg': '#3498db', 'neutral': '#95a5a6', 'pos': '#e74c3c'}
    labels = {'neg': 'Negative RPE', 'neutral': 'Neutral RPE', 'pos': 'Positive RPE'}

    # ========== ROW 1: FREQUENCY BAND POWER ==========

    # Panel 1: Source ROI band power
    ax = axes[0, 0]
    for cat in valid_cats:
        if band_power_results['source'][cat] is not None:
            mean_power = band_power_results['source'][cat]['mean']
            sem = band_power_results['source'][cat]['sem']
            n = band_power_results['source'][cat]['n_trials']
            ax.plot(times, mean_power, color=colors[cat], 
                   label=f"{labels[cat]} (n={n})", linewidth=2)
            ax.fill_between(times, mean_power - sem, mean_power + sem, 
                           color=colors[cat], alpha=0.2)
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Time (s)')
    # Determine band name if not provided
    if band_name is None:
        band_name = get_frequency_band_params(fmin, fmax)['band_name']
    ax.set_ylabel(f'{band_name.capitalize()} Power (a.u.)')
    ax.set_title(f'Source: {roi_source}')
    ax.legend(fontsize=8, loc='upper right')
    # Panel 2: Target ROI band power
    ax = axes[0, 1]
    for cat in valid_cats:
        if band_power_results['target'][cat] is not None:
            mean_power = band_power_results['target'][cat]['mean']
            sem = band_power_results['target'][cat]['sem']
            n = band_power_results['target'][cat]['n_trials']
            ax.plot(times, mean_power, color=colors[cat], 
                   label=f"{labels[cat]} (n={n})", linewidth=2)
            ax.fill_between(times, mean_power - sem, mean_power + sem, 
                           color=colors[cat], alpha=0.2)

    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel(f'{band_name.capitalize()} Power (a.u.)')
    ax.set_title(f'Target: {roi_target}')
    ax.legend(fontsize=8, loc='upper right')

    # Panel 3: Both ROIs overlaid (using neutral RPE as example)

    ax = axes[0, 2]
    roi_colors = {'source': '#2ecc71', 'target': '#9b59b6'}
    for roi_type, roi_name in [('source', roi_source), ('target', roi_target)]:
        if band_power_results[roi_type]['neutral'] is not None:
            mean_power = band_power_results[roi_type]['neutral']['mean']
            sem = band_power_results[roi_type]['neutral']['sem']
            ax.plot(times, mean_power, color=roi_colors[roi_type], 
                   label=f'{roi_name}', linewidth=2)
            ax.fill_between(times, mean_power - sem, mean_power + sem,
                        color=roi_colors[roi_type], alpha=0.2)
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel(f'{band_name.capitalize()} Power (a.u.)')
    ax.set_title('Source vs Target (Neutral RPE)')
    ax.legend(fontsize=8, loc='upper right')

    # ========== ROW 2: GRANGER CAUSALITY ==========
    # Panel 4: Forward GC (source -> target)

    ax = axes[1, 0]
    for cat in valid_cats:
        gc_data = combined_gc[cat]
        n_hemi = gc_data.get('n_hemispheres', 1)
        label_suffix = f" ({n_hemi}H)" if n_hemi > 1 else ""
        ax.plot(times, gc_data['gc_forward'], color=colors[cat], 
               label=f"{labels[cat]}{label_suffix}", linewidth=2)

    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('GC')
    ax.set_title(f'Forward GC: {roi_source} → {roi_target}')
    ax.legend(fontsize=8, loc='upper right')

    # Panel 5: Backward GC (target -> source)
    ax = axes[1, 1]
    for cat in valid_cats:
        gc_data = combined_gc[cat]
        ax.plot(times, gc_data['gc_backward'], color=colors[cat], 
               label=labels[cat], linewidth=2)

    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('GC')
    ax.set_title(f'Backward GC: {roi_target} → {roi_source}')
    ax.legend(fontsize=8, loc='upper right')

    # Panel 6: Net GC (forward - backward)
    ax = axes[1, 2]

    for cat in valid_cats:
        gc_data = combined_gc[cat]
        ax.plot(times, gc_data['gc_net'], color=colors[cat], 
               label=labels[cat], linewidth=2)
        ax.fill_between(times, 0, gc_data['gc_net'], alpha=0.2, color=colors[cat])

    ax.axhline(0, color='black', linestyle='-', linewidth=1)
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Net GC (Fwd - Bwd)')
    ax.set_title(f'Net GC: {roi_source} ↔ {roi_target}')
    ax.legend(fontsize=8, loc='upper right')

    # Overall title
    # Determine band name if not provided
    if band_name is None:
        band_name = get_frequency_band_params(fmin, fmax)['band_name']

    fig.suptitle(f'{subj_id}: {roi_source} → {roi_target} | {band_name.capitalize()} ({fmin}-{fmax} Hz)', 
                 fontsize=14, fontweight='bold', y=1.02)

    plt.tight_layout()

    # Save if path provided
    if savepath is not None:
        plot_file = f'{plot_dir}/{subj_id}_{roi_source}_{roi_target}_gc_plot.png'
        plt.savefig(plot_file, dpi=150, bbox_inches='tight')
        print(f"  Saved plot: {plot_file}")

    return fig

print("Helper functions defined: export_gc_to_dataframe, generate_subject_plot")

In [ ]:
def get_frequency_band_params(fmin, fmax):
    """
    Automatically determine appropriate parameters for a given frequency band.

    Parameters
    ----------
    fmin, fmax : float
        Frequency band limits (Hz)

    Returns
    -------
    dict : Dictionary with recommended parameters:
        - target_sfreq: Sampling frequency (Hz)
        - gc_n_lags: Number of lags for state-space model
        - n_cycles: Number of cycles for wavelet
        - n_freqs: Number of frequency bins
        - band_name: Name of frequency band
        - lag_time_ms: Lag time in milliseconds
        - period_ms: Period of center frequency in milliseconds
    """
    bandwidth = fmax - fmin
    center_freq = (fmin + fmax) / 2

    # Determine band name
    if 4 <= fmin < 8 and 4 < fmax <= 8:
        band_name = 'theta'
    elif 8 <= fmin < 13 and 8 < fmax <= 13:
        band_name = 'alpha'
    elif 13 <= fmin < 30 and 13 < fmax <= 30:
        band_name = 'beta'
    elif 30 <= fmin < 100:
        band_name = 'gamma'
    else:
        band_name = f'{fmin:.0f}-{fmax:.0f}Hz'

    # Determine sampling frequency (must be > 2 * fmax for Nyquist)
    # Use higher sampling rate for higher frequencies
    if fmax <= 13:
        target_sfreq = 50  # Good for theta and alpha
    elif fmax <= 30:
        target_sfreq = 100  # Required for beta (Nyquist = 50 Hz > 30 Hz)
    else:
        target_sfreq = 200  # For gamma and higher

    # Determine number of lags based on period
    # Want to capture ~1-2 cycles of the center frequency
    period_ms = 1000 / center_freq  # Period in milliseconds

    # Use 40-60% of period as lag window
    lag_time_ms = period_ms * 0.5
    gc_n_lags = max(3, int(np.ceil(lag_time_ms * target_sfreq / 1000)))

    # Constraint: gc_n_lags < 2 * n_freqs (from mne_connectivity)
    # We'll set n_freqs first, then adjust gc_n_lags if needed
    if bandwidth < 10:
        n_freqs = 5
    elif bandwidth < 20:
        n_freqs = 8
    else:
        n_freqs = 10

    # Ensure constraint is satisfied
    max_lags = 2 * n_freqs - 1
    if gc_n_lags >= max_lags:
        gc_n_lags = max(3, max_lags - 1)

    # Number of cycles for wavelet (fewer for faster oscillations)
    if center_freq < 10:
        n_cycles = 5
    elif center_freq < 20:
        n_cycles = 4
    else:
        n_cycles = 3

    return {
        'target_sfreq': target_sfreq,
        'gc_n_lags': gc_n_lags,
        'n_cycles': n_cycles,
        'n_freqs': n_freqs,
        'band_name': band_name,
        'lag_time_ms': gc_n_lags / target_sfreq * 1000,
        'period_ms': period_ms
    }

    print("Frequency band parameter helper function defined")

In [ ]:
## High-Power Electrode Visualization Functions

# Functions to identify and visualize electrodes with high power in a frequency band of interest.

In [ ]:
def plot_averaged_tfr_high_power(subj_id, base_dir, high_power_electrodes, event, fmin, fmax,
                                  tfr_time_window=None, day='Day1', figsize=None):
    """
    Plot averaged TFR for high-power electrodes.

    Parameters
    ----------
    subj_id : str
        Subject ID
    base_dir : str
        Base directory path
    high_power_electrodes : list
        List of electrode names from identify_high_power_electrodes()
    event : str
        Event name (e.g., 'feedback_start')
    fmin, fmax : float
        Frequency band of interest (Hz)
    tfr_time_window : tuple, optional
        (tmin, tmax) for time averaging, None = all times
    day : str
        Day folder (default: 'Day1')
    figsize : tuple, optional
        Figure size (width, height). If None, auto-calculated.

    Returns
    -------
    matplotlib.figure.Figure : The figure object, or None if TFR file not found
    """
    # Load TFR file
    tfr_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR/{event}-tfr.h5'
    
    if not os.path.exists(tfr_path):
        print(f"  TFR file not found: {tfr_path}")
        return None

    try:
        tfrs = mne.time_frequency.read_tfrs(tfr_path, verbose=False)
        tfr = tfrs[0]  # Get first TFR object
    except Exception as e:
        print(f"  Error loading TFR: {e}")
        return None

    # Filter electrodes to those that exist in TFR
    available_electrodes = [ch for ch in high_power_electrodes if ch in tfr.ch_names]
    
    if len(available_electrodes) == 0:
        print(f"  No high-power electrodes found in TFR data")
        return None

    # Determine figure size
    n_electrodes = len(available_electrodes)
    if figsize is None:
        n_cols = min(4, n_electrodes)
        n_rows = int(np.ceil(n_electrodes / n_cols))
        figsize = (5 * n_cols, 4 * n_rows)
    else:
        n_cols = min(4, n_electrodes)
        n_rows = int(np.ceil(n_electrodes / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    if n_electrodes == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    # Plot each electrode
    for idx, ch_name in enumerate(available_electrodes):
        ax = axes[idx]
        
        # Pick single channel
        tfr_ch = tfr.copy().pick_channels([ch_name])
        
        # Average across trials
        tfr_avg = tfr_ch.average()
        
        # Apply time window if specified
        if tfr_time_window is not None:
            tfr_avg = tfr_avg.crop(tmin=tfr_time_window[0], tmax=tfr_time_window[1])
        
        # Get data: (n_freqs, n_times) after averaging across trials
        tfr_data = tfr_avg.data[0, :, :]  # Remove channel dimension
        
        # Get frequencies and times
        freqs = tfr_avg.freqs
        times = tfr_avg.times
        
        # Create frequency mask for band of interest
        freq_mask = (freqs >= fmin) & (freqs <= fmax)
        
        # Plot time-frequency representation
        im = ax.imshow(tfr_data, aspect='auto', origin='lower',
                      extent=[times[0], times[-1], freqs[0], freqs[-1]],
                      cmap='viridis', interpolation='nearest')
        
        # Highlight frequency band
        ax.axhspan(fmin, fmax, alpha=0.3, color='red', label=f'{fmin}-{fmax} Hz')
        
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Frequency (Hz)')
        ax.set_title(f'{ch_name}\nAveraged TFR (across trials)')
        ax.legend(fontsize=8, loc='upper right')
        
        # Add colorbar
        plt.colorbar(im, ax=ax, label='Power (a.u.)')

    # Hide unused subplots
    for idx in range(len(available_electrodes), len(axes)):
        axes[idx].axis('off')

    plt.suptitle(f'{subj_id}: Averaged TFR for High-Power Electrodes ({fmin}-{fmax} Hz)',
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()

    return fig


def identify_high_power_electrodes(epoch_data, fmin, fmax, percentile_threshold=50, roi=None):
    """
    Identify electrodes in the top X percentile of power within a frequency band.

    Parameters
    ----------
    epoch_data : dict
        Output from load_subject_epochs()
    fmin, fmax : float
        Frequency band of interest (Hz)
    percentile_threshold : float
        Percentile cutoff (default: 50 for top 50%)
    roi : str, optional
        ROI filter - if None, checks all electrodes. If specified, only checks
        electrodes in that ROI (e.g., 'ACC', 'OFC')

    Returns
    -------
    dict : Dictionary containing:
        - 'electrodes': List of electrode names in top percentile
        - 'power_values': Dictionary mapping electrode -> mean power value
        - 'percentile_ranks': Dictionary mapping electrode -> percentile rank
        - 'threshold_power': Power value at percentile threshold
    """
    epochs = epoch_data['epochs']

    # Get electrodes to check
    if roi is not None:
        # Filter by ROI - check both source and target ROIs
        all_channels = []
        for roi_type in ['source', 'target']:
            for hemi in ['l', 'r']:
                channels = epoch_data.get(f'{roi_type}_channels_{hemi}', [])
                all_channels.extend(channels)
        
        # Also check if roi matches source or target from epoch_data
        # This is a simple check - in practice you'd want to match against elec_df
        electrodes_to_check = [ch for ch in epochs.ch_names if ch in all_channels]
    else:
        electrodes_to_check = epochs.ch_names

    if len(electrodes_to_check) == 0:
        return {
            'electrodes': [],
            'power_values': {},
            'percentile_ranks': {},
            'threshold_power': 0.0
        }

    # Compute power for each electrode
    electrode_powers = {}
    
    for ch_name in electrodes_to_check:
        # Pick single channel
        epochs_ch = epochs.copy().pick_channels([ch_name])
        
        # Filter to frequency band
        epochs_ch = epochs_ch.copy().filter(fmin, fmax, verbose=False)
        
        # Compute power (squared amplitude) and average across time and trials
        power_data = epochs_ch.get_data() ** 2  # (n_epochs, 1, n_times)
        mean_power = np.mean(power_data)  # Average across all dimensions
        electrode_powers[ch_name] = mean_power

    # Calculate percentile threshold
    power_values = list(electrode_powers.values())
    threshold_power = np.percentile(power_values, 100 - percentile_threshold)

    # Find electrodes above threshold
    high_power_electrodes = [
        ch for ch, power in electrode_powers.items()
        if power >= threshold_power
    ]

    # Calculate percentile ranks
    percentile_ranks = {}
    for ch, power in electrode_powers.items():
        percentile_ranks[ch] = (np.sum(np.array(power_values) <= power) / len(power_values)) * 100

    return {
        'electrodes': high_power_electrodes,
        'power_values': electrode_powers,
        'percentile_ranks': percentile_ranks,
        'threshold_power': threshold_power
    }


def plot_raw_lfp_high_power_trials(epoch_data, high_power_electrodes, fmin, fmax,
                                    n_trials=5, tmin=None, tmax=None):
    """
    Plot raw LFP traces from individual high-power trials.

    Parameters
    ----------
    epoch_data : dict
        Output from load_subject_epochs()
    high_power_electrodes : list
        List of electrode names
    fmin, fmax : float
        Frequency band of interest (Hz)
    n_trials : int
        Number of top trials to plot per electrode (default: 5)
    tmin, tmax : float, optional
        Time window for plotting. If None, uses full epoch window.

    Returns
    -------
    dict : Dictionary mapping electrode name -> matplotlib.figure.Figure
    """
    epochs = epoch_data['epochs']
    times = epoch_data['times']

    # Crop time window if specified
    if tmin is not None or tmax is not None:
        if tmin is None:
            tmin = times[0]
        if tmax is None:
            tmax = times[-1]
        time_mask = (times >= tmin) & (times <= tmax)
        times_plot = times[time_mask]
    else:
        times_plot = times
        time_mask = slice(None)

    figures = {}

    for ch_name in high_power_electrodes:
        if ch_name not in epochs.ch_names:
            continue

        # Pick single channel
        epochs_ch = epochs.copy().pick_channels([ch_name])

        # Get unfiltered data first (for overlay)
        epochs_unfiltered = epochs_ch.copy()
        data_unfiltered = epochs_unfiltered.get_data()  # (n_epochs, 1, n_times)

        # Filter to frequency band
        epochs_ch = epochs_ch.copy().filter(fmin, fmax, verbose=False)

        # Get filtered data: (n_epochs, 1, n_times)
        data_filtered = epochs_ch.get_data()

        # Compute power for each trial (mean across time) using filtered data
        if time_mask == slice(None):
            trial_powers = np.mean(data_filtered[:, 0, :] ** 2, axis=1)
        else:
            trial_powers = np.mean(data_filtered[:, 0, time_mask] ** 2, axis=1)

        # Get top N trials
        top_trial_indices = np.argsort(trial_powers)[-n_trials:][::-1]  # Descending order

        # Create figure with subplots
        fig, axes = plt.subplots(n_trials, 1, figsize=(12, 2 * n_trials), sharex=True)

        if n_trials == 1:
            axes = [axes]

        # Plot each trial
        for idx, trial_idx in enumerate(top_trial_indices):
            ax = axes[idx]

            # Get unfiltered and filtered trial data
            if time_mask == slice(None):
                trial_data_unfiltered = data_unfiltered[trial_idx, 0, :]
                trial_data_filtered = data_filtered[trial_idx, 0, :]
            else:
                trial_data_unfiltered = data_unfiltered[trial_idx, 0, time_mask]
                trial_data_filtered = data_filtered[trial_idx, 0, time_mask]

            # Plot unfiltered data in light gray (background)
            ax.plot(times_plot, trial_data_unfiltered, linewidth=1.0, color='gray', alpha=0.5, label='Raw LFP')
            
            # Plot filtered data in black (foreground)
            ax.plot(times_plot, trial_data_filtered, linewidth=1.5, color='black', label='Filtered LFP')
            
            ax.axvline(0, color='red', linestyle='--', alpha=0.5, linewidth=1)
            ax.set_ylabel('Amplitude (μV)', fontsize=9)
            ax.set_title(f'Trial {trial_idx} | Power: {trial_powers[trial_idx]:.4f}',
                        fontsize=10, fontweight='bold')
            ax.grid(True, alpha=0.3)
            
            # Add legend only to first subplot
            if idx == 0:
                ax.legend(fontsize=8, loc='upper right')

        # Set x-axis label on bottom subplot
        axes[-1].set_xlabel('Time (s)', fontsize=10)

        # Overall title
        fig.suptitle(f'{ch_name}: Top {n_trials} High-Power Trials ({fmin}-{fmax} Hz)',
                    fontsize=12, fontweight='bold', y=1.02)

        plt.tight_layout()
        figures[ch_name] = fig

    return figures


def visualize_high_power_electrodes(subj_id, base_dir, epoch_data, event, fmin, fmax,
                                     percentile_threshold=50, roi=None, n_trials=5,
                                     tfr_time_window=None, tmin=None, tmax=None,
                                     day='Day1', savepath=None):
    """
    Convenience wrapper function that identifies high-power electrodes and creates visualizations.

    Parameters
    ----------
    subj_id : str
        Subject ID
    base_dir : str
        Base directory path
    epoch_data : dict
        Output from load_subject_epochs()
    event : str
        Event name (e.g., 'feedback_start')
    fmin, fmax : float
        Frequency band of interest (Hz)
    percentile_threshold : float
        Percentile cutoff for high-power electrodes (default: 50 for top 50%)
    roi : str, optional
        ROI filter (if None, checks all electrodes)
    n_trials : int
        Number of top trials to plot per electrode for LFP traces (default: 5)
    tfr_time_window : tuple, optional
        (tmin, tmax) for TFR time averaging, None = all times
    tmin, tmax : float, optional
        Time window for LFP plotting. If None, uses full epoch window.
    day : str
        Day folder (default: 'Day1')
    savepath : str, optional
        If provided, save plots to this directory

    Returns
    -------
    dict : Dictionary containing:
        - 'high_power_info': Output from identify_high_power_electrodes()
        - 'tfr_figure': Figure from plot_averaged_tfr_high_power() or None
        - 'lfp_figures': Dictionary of figures from plot_raw_lfp_high_power_trials()
    """
    # Step 1: Identify high-power electrodes
    print(f"Identifying high-power electrodes (top {percentile_threshold}%)...")
    high_power_info = identify_high_power_electrodes(
        epoch_data, fmin, fmax,
        percentile_threshold=percentile_threshold,
        roi=roi
    )

    high_power_electrodes = high_power_info['electrodes']
    n_electrodes = len(high_power_electrodes)

    if n_electrodes == 0:
        print(f"  No high-power electrodes found!")
        return {
            'high_power_info': high_power_info,
            'tfr_figure': None,
            'lfp_figures': {}
        }

    print(f"  Found {n_electrodes} high-power electrodes: {high_power_electrodes}")

    # Step 2: Plot averaged TFR
    print(f"\nPlotting averaged TFR...")
    tfr_figure = plot_averaged_tfr_high_power(
        subj_id, base_dir, high_power_electrodes, event, fmin, fmax,
        tfr_time_window=tfr_time_window, day=day
    )

    if tfr_figure is not None and savepath is not None:
        tfr_file = f'{plot_dir}/{subj_id}_high_power_tfr_{fmin}-{fmax}Hz.png'
        tfr_figure.savefig(tfr_file, dpi=150, bbox_inches='tight')
        print(f"  Saved TFR plot: {tfr_file}")

    # Step 3: Plot raw LFP traces
    print(f"\nPlotting raw LFP traces for top {n_trials} trials per electrode...")
    lfp_figures = plot_raw_lfp_high_power_trials(
        epoch_data, high_power_electrodes, fmin, fmax,
        n_trials=n_trials, tmin=tmin, tmax=tmax
    )

    if savepath is not None:
        for ch_name, fig in lfp_figures.items():
            lfp_file = f'{plot_dir}/{subj_id}_{ch_name}_high_power_lfp_{fmin}-{fmax}Hz.png'
            fig.savefig(lfp_file, dpi=150, bbox_inches='tight')
            print(f"  Saved LFP plot: {lfp_file}")

    print(f"\nVisualization complete!")

    return {
        'high_power_info': high_power_info,
        'tfr_figure': tfr_figure,
        'lfp_figures': lfp_figures
    }


print("High-power electrode visualization functions defined")

In [ ]:
# Example usage: Visualize high-power electrodes
# This example shows how to identify and plot electrodes with high power in the frequency band of interest

# Example parameters (adjust based on your analysis)
example_subj_id = 'MS012'  # Change to your subject of interest
example_roi_source = 'ACC'
example_roi_target = 'OFC'
example_fmin = 4  # Hz
example_fmax = 8  # Hz
example_percentile = 50  # Top 50% of electrodes
example_n_trials = 5  # Number of high-power trials to plot per electrode

# Load epoch data for the subject
example_epoch_data = load_subject_epochs(
    example_subj_id, base_dir, analysis_params['event'],
    example_roi_source, example_roi_target,
    tmin=analysis_params['tmin'],
    tmax=analysis_params['tmax'],
    target_sfreq=500
)

if example_epoch_data is not None:
    # Option 1: Use the wrapper function (recommended - does everything)
    results = visualize_high_power_electrodes(
        subj_id=example_subj_id,
        base_dir=base_dir,
        epoch_data=example_epoch_data,
        event=analysis_params['event'],
        fmin=example_fmin,
        fmax=example_fmax,
        percentile_threshold=example_percentile,
        roi=None,  # Set to 'ACC' or 'OFC' to filter by ROI
        n_trials=example_n_trials,
        tfr_time_window=None,  # Or specify (0.0, 1.5) for time window
        tmin=None,  # Or specify time window for LFP plots
        tmax=None,
        savepath=savepath  # Optional: save plots
    )
    
    # Display results
    print(f"\nHigh-power electrodes: {results['high_power_info']['electrodes']}")
    print(f"Power values: {results['high_power_info']['power_values']}")
    
    # Show plots
    if results['tfr_figure'] is not None:
        plt.show(results['tfr_figure'])
    
    for ch_name, fig in results['lfp_figures'].items():
        plt.show(fig)
    
    # Option 2: Use functions individually for more control
    # Step 1: Identify high-power electrodes
    # high_power_info = identify_high_power_electrodes(
    #     example_epoch_data, example_fmin, example_fmax,
    #     percentile_threshold=example_percentile,
    #     roi=None
    # )
    # 
    # # Step 2: Plot TFR
    # if len(high_power_info['electrodes']) > 0:
    #     tfr_fig = plot_averaged_tfr_high_power(
    #         example_subj_id, base_dir, high_power_info['electrodes'],
    #         analysis_params['event'], example_fmin, example_fmax
    #     )
    #     if tfr_fig is not None:
    #         plt.show(tfr_fig)
    # 
    # # Step 3: Plot raw LFP traces
    # lfp_figs = plot_raw_lfp_high_power_trials(
    #     example_epoch_data, high_power_info['electrodes'],
    #     example_fmin, example_fmax, n_trials=example_n_trials
    # )
    # for ch_name, fig in lfp_figs.items():
    #     plt.show(fig)
else:
    print(f"Could not load epoch data for {example_subj_id}")

## Analysis Parameters

### Frequency Band Configuration

**To analyze different frequency bands, simply change `fmin_user` and `fmax_user` in the cell below.**

The code will automatically adjust:
- **Sampling rate** (`target_sfreq`): Higher for higher frequencies
- **Number of lags** (`gc_n_lags`): Adjusted to capture 1-2 cycles of the center frequency
- **Number of frequency bins** (`n_freqs`): More bins for wider bandwidths
- **Wavelet cycles** (`n_cycles`): Fewer cycles for faster oscillations

**Common frequency bands:**
- **Theta**: 4-8 Hz
- **Alpha**: 8-13 Hz
- **Beta**: 13-30 Hz
- **Gamma**: 30-100 Hz

**Example:** To analyze alpha band, change:
```python
fmin_user = 8   # Hz
fmax_user = 13  # Hz
```

In [ ]:
# Analysis parameters for multivariate spectral Granger causality
# Set frequency band - parameters will auto-adjust
fmin_user = 4   # Hz - Change this for different bands (theta: 4-8, alpha: 8-13, beta: 13-30)
fmax_user = 8   # Hz

# Get auto-adjusted parameters based on frequency band
band_params = get_frequency_band_params(fmin_user, fmax_user)

analysis_params = {
    # ROI pairs to analyze (source, target)
    # Source: ACC, Targets: dmPFC, OFC, dlPFC, HPC, AMY
    'roi_pairs': [
        ('ACC', 'dmPFC'),
        ('ACC', 'OFC'),
        ('ACC', 'dlPFC'),
        ('ACC', 'HPC'),
        ('ACC', 'AMY'),
    ],
    # Event to analyze
    'event': 'feedback_start',
    # Time window (seconds relative to event)
    'tmin': 0.0,
    'tmax': 1.5,
    # Frequency band
    'fmin': fmin_user,
    'fmax': fmax_user,
    # Auto-adjusted parameters based on frequency band
    'target_sfreq': band_params['target_sfreq'],
    'gc_n_lags': band_params['gc_n_lags'],
    'n_cycles': band_params['n_cycles'],
    'n_freqs': band_params['n_freqs'],
    'band_name': band_params['band_name'],
    # Electrode filtering by band power (optional)
    'filter_electrodes_by_power': False,  # Enable/disable filtering
    'power_percentile_threshold': 25,     # Keep top X% per ROI (50 = top 50%)
    'tfr_time_window': None,              # (tmin, tmax) for power averaging, None = all times
    
    # Top electrode selection (optional)
    'use_top_electrode_only': True,      # If True, use only top electrode per ROI per hemisphere
                                          # based on median power in frequency band
}

# Print configuration
print(f"Frequency band configuration:")
print(f"  Band: {band_params['band_name']} ({analysis_params['fmin']}-{analysis_params['fmax']} Hz)")
print(f"  Target sampling rate: {analysis_params['target_sfreq']} Hz")
print(f"  gc_n_lags: {analysis_params['gc_n_lags']}")
print(f"  Maximum lag: {band_params['lag_time_ms']:.0f} ms")
print(f"  Center frequency period: {band_params['period_ms']:.0f} ms")
print(f"  Number of frequency bins: {analysis_params['n_freqs']}")
print(f"  Wavelet cycles: {analysis_params['n_cycles']}")
print(f"  Electrode power filtering: {analysis_params.get('filter_electrodes_by_power', False)} (top {analysis_params.get('power_percentile_threshold', 50)}% per ROI)")
print(f"  Use top electrode only: {analysis_params.get('use_top_electrode_only', False)}")

# Save path for results
savepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/GrangerCausality_Spectral_{band_params["band_name"]}'
plot_dir = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit-MultivariateGrangerCausality'
os.makedirs(savepath, exist_ok=True)
os.makedirs(plot_dir, exist_ok=True)

print(f"Results will be saved to: {savepath}")

## Main Analysis Loop

In [ ]:
# Main Analysis Loop
# Process all subjects for all ROI pairs

all_results = {}
print(f"\n{'='*70}")
print("Starting Multivariate Spectral Granger Causality Analysis")
print(f"{'='*70}")
print(f"Subjects: {len(subj_ids)}")
print(f"ROI pairs: {len(analysis_params['roi_pairs'])}")
band_name = analysis_params.get('band_name', 'frequency band')
print(f"Frequency: {analysis_params['fmin']}-{analysis_params['fmax']} Hz ({band_name})")
print(f"Downsampling to: {analysis_params['target_sfreq']} Hz")
print(f"Max lag: {analysis_params['gc_n_lags'] / analysis_params['target_sfreq'] * 1000:.0f} ms")
print(f"{'='*70}\n")

for roi_source, roi_target in analysis_params['roi_pairs']:
    pair_key = f'{roi_source}_{roi_target}'
    print(f"\n{'='*60}")
    print(f"Processing: {roi_source} -> {roi_target}")
    print(f"{'='*60}")
    pair_results = []

    for subj_id in tqdm(subj_ids, desc=f'{roi_source}->{roi_target}'):
        # Process subject
        result = process_subject_gc(
            subj_id, roi_source, roi_target, base_dir, learn_df,
            event=analysis_params['event'],
            tmin=analysis_params['tmin'],
            tmax=analysis_params['tmax'],
            target_sfreq=analysis_params['target_sfreq'],
            fmin=analysis_params['fmin'],
            fmax=analysis_params['fmax'],
            gc_n_lags=analysis_params['gc_n_lags'],
            n_freqs=analysis_params['n_freqs'],
            verbose=False,
            filter_by_power=analysis_params.get('filter_electrodes_by_power', False),
            power_percentile=analysis_params.get('power_percentile_threshold', 50),
            tfr_time_window=analysis_params.get('tfr_time_window', None),
            use_top_electrode_only=analysis_params.get('use_top_electrode_only', False)
        )

        if result is not None:
            pair_results.append(result)

            # Load epoch data for plotting
            epoch_data = load_subject_epochs(
                subj_id, base_dir, analysis_params['event'],
                roi_source, roi_target,
                tmin=analysis_params['tmin'],
                tmax=analysis_params['tmax'],
                target_sfreq=analysis_params['target_sfreq'],
                filter_by_power=analysis_params.get('filter_electrodes_by_power', False),
                power_percentile=analysis_params.get('power_percentile_threshold', 50),
                fmin=analysis_params['fmin'],
                fmax=analysis_params['fmax'],
                tfr_time_window=analysis_params.get('tfr_time_window', None),
                use_top_electrode_only=analysis_params.get('use_top_electrode_only', False)
            )
            if epoch_data is not None:
                # Generate and save plot
                try:
                    fig = generate_subject_plot(
                        subj_id, roi_source, roi_target, result, epoch_data,
                        learn_df,
                        fmin=analysis_params['fmin'],
                        fmax=analysis_params['fmax'],
                        band_name=analysis_params.get('band_name'),
                        savepath=savepath
                    )
                    if fig is not None:
                        plt.close(fig)  # Close to free memory
                except Exception as e:
                    print(f"    Warning: Could not generate plot for {subj_id}: {e}")
                # Export to CSV
                try:
                    subj_df = export_gc_to_dataframe(result, subj_id, roi_source, roi_target)
                    csv_file = f'{savepath}/{subj_id}_{roi_source}_{roi_target}_gc_results.csv'
                    subj_df.to_csv(csv_file, index=False)
                except Exception as e:
                    print(f"    Warning: Could not export CSV for {subj_id}: {e}")
    
    all_results[pair_key] = pair_results
    # Summary
    n_success = len(pair_results)
    print(f"  Completed: {n_success}/{len(subj_ids)} subjects")

    # Save intermediate results
    import pickle
    save_file = f'{savepath}/{pair_key}_spectral_gc.pkl'
    with open(save_file, 'wb') as f:
        pickle.dump(pair_results, f)
    print(f"  Saved to: {save_file}")

print(f"\n{'='*70}")
print("Analysis complete!")
print(f"{'='*70}")

In [ ]:
# =============================================================================

# LOAD ALL SAVED RESULTS

# =============================================================================

# Load all pickle files and combine into a single structure



import pickle

import glob



def load_all_results(savepath):

    """

    Load all saved GC results from pickle files.

    

    Parameters

    ----------

    savepath : str

        Base path where results are saved

    

    Returns

    -------

    dict : Dictionary with all results, keyed by ROI pair

    """

    all_results = {}

    

    # Find all pickle files

    pickle_files = glob.glob(f'{savepath}/*_spectral_gc.pkl')

    

    print(f"Found {len(pickle_files)} result files")

    

    for pkl_file in pickle_files:

        # Extract ROI pair from filename

        basename = os.path.basename(pkl_file)

        pair_key = basename.replace('_spectral_gc.pkl', '')

        

        try:

            with open(pkl_file, 'rb') as f:

                results = pickle.load(f)

            

            all_results[pair_key] = results

            

            # Print summary

            n_subjects = len(results)

            print(f"  {pair_key}: {n_subjects} subjects")

            

        except Exception as e:

            print(f"  Warning: Could not load {pkl_file}: {e}")

    

    return all_results





# Load all results

print("Loading all saved results...")

all_results = load_all_results(savepath)



# Print summary

print(f"\n{'='*60}")

print("Summary of loaded results:")

print(f"{'='*60}")

for pair_key, results in all_results.items():

    subj_ids = [r['participant'] for r in results]

    print(f"{pair_key}: {len(subj_ids)} subjects")

    print(f"  Subjects: {', '.join(subj_ids)}")



print(f"\nTotal ROI pairs: {len(all_results)}")

print(f"Total subjects across all pairs: {sum(len(r) for r in all_results.values())}")

In [ ]:
# =============================================================================

# AGGREGATED PLOTS ACROSS SUBJECTS

# =============================================================================

# Create plots showing mean ± SEM across subjects for each target ROI



def create_aggregated_plots(all_results, savepath, fmin=4, fmax=8, band_name=None):

    """

    Create aggregated plots across all subjects.

    

    Parameters

    ----------

    all_results : dict

        Dictionary of results from load_all_results()

    savepath : str

        Path to save plots

    fmin, fmax : float

        Frequency band for title

    """

    rpe_cats = ['neg', 'neutral', 'pos']

    colors = {'neg': '#3498db', 'neutral': '#95a5a6', 'pos': '#e74c3c'}

    labels = {'neg': 'Negative RPE', 'neutral': 'Neutral RPE', 'pos': 'Positive RPE'}

    

    # Extract target ROIs from pair keys

    target_rois = sorted(set([key.split('_')[1] for key in all_results.keys()]))

    

    # ========== PLOT 1: Time-resolved GC by Target ROI ==========

    fig, axes = plt.subplots(len(target_rois), 3, figsize=(18, 5*len(target_rois)))

    if len(target_rois) == 1:

        axes = axes.reshape(1, -1)

    

    for tgt_idx, target_roi in enumerate(target_rois):

        # Find all pairs with this target

        pair_keys = [k for k in all_results.keys() if k.endswith(f'_{target_roi}')]

        

        if len(pair_keys) == 0:

            continue

        

        # Use first pair to get times (should be same for all)

        first_pair = all_results[pair_keys[0]]

        if len(first_pair) == 0:

            continue

        

        # Get times from first subject

        combined_gc = combine_hemispheres(first_pair[0]['gc_by_hemi'])

        times = None

        for cat in rpe_cats:

            if cat in combined_gc and combined_gc[cat].get('success', False):

                times = combined_gc[cat]['times']

                break

        

        if times is None:

            continue

        

        # Collect GC values across subjects

        gc_by_cat = {cat: {'forward': [], 'backward': [], 'net': []} for cat in rpe_cats}

        

        for pair_key in pair_keys:

            for result in all_results[pair_key]:

                combined_gc = combine_hemispheres(result['gc_by_hemi'])

                for cat in rpe_cats:

                    if cat in combined_gc and combined_gc[cat].get('success', False):

                        gc_data = combined_gc[cat]

                        gc_by_cat[cat]['forward'].append(gc_data['gc_forward'])

                        gc_by_cat[cat]['backward'].append(gc_data['gc_backward'])

                        gc_by_cat[cat]['net'].append(gc_data['gc_net'])

        

        # Compute mean and SEM

        for gc_type, ax in zip(['forward', 'backward', 'net'], axes[tgt_idx]):

            for cat in rpe_cats:

                if len(gc_by_cat[cat][gc_type]) > 0:

                    gc_array = np.array(gc_by_cat[cat][gc_type])  # (n_subjects, n_times)

                    mean_gc = np.mean(gc_array, axis=0)

                    sem_gc = np.std(gc_array, axis=0) / np.sqrt(len(gc_array))

                    

                    ax.plot(times, mean_gc, color=colors[cat], 

                           label=labels[cat], linewidth=2)

                    ax.fill_between(times, mean_gc - sem_gc, mean_gc + sem_gc,

                                   color=colors[cat], alpha=0.2)

            

            ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

            ax.axvline(0, color='gray', linestyle='--', alpha=0.5)

            ax.set_xlabel('Time (s)')

            ax.set_ylabel('GC')

            

            if gc_type == 'forward':

                ax.set_title(f'{target_roi}: Forward GC (ACC → {target_roi})')

            elif gc_type == 'backward':

                ax.set_title(f'{target_roi}: Backward GC ({target_roi} → ACC)')

            else:

                ax.set_title(f'{target_roi}: Net GC')

            

            ax.legend(fontsize=8, loc='upper right')

    

    # Determine band name if not provided

    if band_name is None:

        band_name = get_frequency_band_params(fmin, fmax)['band_name']

    

    fig.suptitle(f'Aggregated GC Across Subjects | {band_name.capitalize()} ({fmin}-{fmax} Hz)', 

                 fontsize=16, fontweight='bold', y=1.0)

    plt.tight_layout()

    

    plot_file = f'{plot_dir}/aggregated_gc_timeseries.png'

    plt.savefig(plot_file, dpi=150, bbox_inches='tight')

    print(f"Saved: {plot_file}")

    plt.show()

    

    # ========== PLOT 2: Bar Plots - Mean GC by Target ROI ==========

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    

    # Collect mean GC values (averaged across time)

    bar_data = {cat: {'forward': [], 'backward': [], 'net': [], 'targets': []} 

                for cat in rpe_cats}

    

    for target_roi in target_rois:

        pair_keys = [k for k in all_results.keys() if k.endswith(f'_{target_roi}')]

        

        for pair_key in pair_keys:

            for result in all_results[pair_key]:

                combined_gc = combine_hemispheres(result['gc_by_hemi'])

                for cat in rpe_cats:

                    if cat in combined_gc and combined_gc[cat].get('success', False):

                        gc_data = combined_gc[cat]

                        bar_data[cat]['forward'].append(np.mean(gc_data['gc_forward']))

                        bar_data[cat]['backward'].append(np.mean(gc_data['gc_backward']))

                        bar_data[cat]['net'].append(np.mean(gc_data['gc_net']))

                        bar_data[cat]['targets'].append(target_roi)

    

    # Create bar plots

    x_pos = np.arange(len(target_rois))

    width = 0.25

    

    for gc_idx, (gc_type, ax) in enumerate(zip(['forward', 'backward', 'net'], axes)):

        means_by_target = {tgt: {cat: [] for cat in rpe_cats} for tgt in target_rois}

        

        for cat in rpe_cats:

            for tgt in target_rois:

                values = [bar_data[cat][gc_type][i] 

                         for i, t in enumerate(bar_data[cat]['targets']) if t == tgt]

                if len(values) > 0:

                    means_by_target[tgt][cat] = np.mean(values)

        

        # Plot bars

        for i, cat in enumerate(rpe_cats):

            means = [means_by_target[tgt][cat] if cat in means_by_target[tgt] else 0 

                    for tgt in target_rois]

            ax.bar(x_pos + i*width, means, width, label=labels[cat], color=colors[cat])

        

        ax.set_xlabel('Target ROI')

        ax.set_ylabel('Mean GC')

        ax.set_title(f'Mean {gc_type.capitalize()} GC by Target ROI')

        ax.set_xticks(x_pos + width)

        ax.set_xticklabels(target_rois)

        ax.legend()

        ax.axhline(0, color='black', linestyle='-', linewidth=0.5)

    

    plt.tight_layout()

    plot_file = f'{plot_dir}/aggregated_gc_bars.png'

    plt.savefig(plot_file, dpi=150, bbox_inches='tight')

    print(f"Saved: {plot_file}")

    plt.show()

    

    # ========== PLOT 3: Heatmaps - Net GC across Time and Target ROIs ==========

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    

    for cat_idx, cat in enumerate(rpe_cats):

        ax = axes[cat_idx]

        

        # Collect net GC data

        heatmap_data = []

        target_labels = []

        

        for target_roi in target_rois:

            pair_keys = [k for k in all_results.keys() if k.endswith(f'_{target_roi}')]

            target_net_gc = []

            

            for pair_key in pair_keys:

                for result in all_results[pair_key]:

                    combined_gc = combine_hemispheres(result['gc_by_hemi'])

                    if cat in combined_gc and combined_gc[cat].get('success', False):

                        target_net_gc.append(combined_gc[cat]['gc_net'])

            

            if len(target_net_gc) > 0:

                mean_net_gc = np.mean(target_net_gc, axis=0)

                heatmap_data.append(mean_net_gc)

                target_labels.append(target_roi)

        

        if len(heatmap_data) > 0:

            heatmap_array = np.array(heatmap_data)

            im = ax.imshow(heatmap_array, aspect='auto', cmap='RdBu_r', 

                          vmin=-np.abs(heatmap_array).max(), 

                          vmax=np.abs(heatmap_array).max())

            ax.set_yticks(range(len(target_labels)))

            ax.set_yticklabels(target_labels)

            ax.set_xlabel('Time (s)')

            ax.set_ylabel('Target ROI')

            ax.set_title(f'Net GC: {labels[cat]}')

            

            # Add time axis

            if times is not None:

                n_times = len(times)

                time_ticks = np.linspace(0, n_times-1, 5).astype(int)

                ax.set_xticks(time_ticks)

                ax.set_xticklabels([f'{times[t]:.2f}' for t in time_ticks])

            

            plt.colorbar(im, ax=ax, label='Net GC')

    

    plt.tight_layout()

    plot_file = f'{plot_dir}/aggregated_gc_heatmaps.png'

    plt.savefig(plot_file, dpi=150, bbox_inches='tight')

    print(f"Saved: {plot_file}")

    plt.show()





# Create aggregated plots

if len(all_results) > 0:

    print("\nCreating aggregated plots...")

    create_aggregated_plots(all_results, savepath, 

                           fmin=analysis_params['fmin'], 

                           fmax=analysis_params['fmax'],

                           band_name=analysis_params.get('band_name'))

else:

    print("No results to plot. Run the main analysis loop first.")

An interpretation: As RPE goes from more negative to move positive, ACC theta becomes less predictable to HPC, and more predictive of OFC/dmPFC theta. 

In [ ]:
# =============================================================================

# STATISTICS-READY DATAFRAME

# =============================================================================

# Create DataFrame with mean GC values (averaged across time) for mixed-effects models



def create_statistics_dataframe(all_results, savepath):

    """

    Create a DataFrame suitable for mixed-effects models.

    

    Parameters

    ----------

    all_results : dict

        Dictionary of results from load_all_results()

    savepath : str

        Path to save DataFrame

    

    Returns

    -------

    pd.DataFrame : DataFrame with columns: participant, roi_target, rpe_cat, 

                   mean_gc_forward, mean_gc_backward, mean_gc_net, n_trials

    """

    rows = []

    rpe_cats = ['neg', 'neutral', 'pos']

    

    for pair_key, results in all_results.items():

        # Extract ROI names from pair_key (format: "ACC_dmPFC")

        parts = pair_key.split('_')

        if len(parts) >= 2:

            roi_source = parts[0]

            roi_target = '_'.join(parts[1:])  # Handle multi-part ROI names

        else:

            continue

        

        for result in results:

            subj_id = result['participant']

            

            # Combine hemispheres

            combined_gc = combine_hemispheres(result['gc_by_hemi'])

            

            for cat in rpe_cats:

                if cat in combined_gc and combined_gc[cat].get('success', False):

                    gc_data = combined_gc[cat]

                    

                    # Average across time

                    mean_forward = np.mean(gc_data['gc_forward'])

                    mean_backward = np.mean(gc_data['gc_backward'])

                    mean_net = np.mean(gc_data['gc_net'])

                    n_trials = gc_data['n_trials']

                    

                    rows.append({

                        'participant': subj_id,

                        'roi_source': roi_source,

                        'roi_target': roi_target,

                        'rpe_cat': cat,

                        'mean_gc_forward': mean_forward,

                        'mean_gc_backward': mean_backward,

                        'mean_gc_net': mean_net,

                        'n_trials': n_trials

                    })

    

    df = pd.DataFrame(rows)

    

    # Print summary

    print(f"Statistics DataFrame created:")

    print(f"  Total rows: {len(df)}")

    print(f"  Subjects: {df['participant'].nunique()}")

    print(f"  Target ROIs: {df['roi_target'].nunique()}")

    print(f"  RPE categories: {df['rpe_cat'].nunique()}")

    print(f"\nColumns: {list(df.columns)}")

    print(f"\nFirst few rows:")

    print(df.head(10))

    

    # Save

    csv_file = f'{savepath}/gc_statistics_dataframe.csv'

    df.to_csv(csv_file, index=False)

    print(f"\nSaved to: {csv_file}")

    

    # Print summary statistics

    print(f"\n{'='*60}")

    print("Summary Statistics by Target ROI and RPE Category:")

    print(f"{'='*60}")

    summary = df.groupby(['roi_target', 'rpe_cat']).agg({

        'mean_gc_net': ['mean', 'std', 'count'],

        'mean_gc_forward': 'mean',

        'mean_gc_backward': 'mean'

    }).round(4)

    print(summary)

    

    return df





# Create statistics dataframe

if len(all_results) > 0:

    print("Creating statistics-ready DataFrame...")

    stats_df = create_statistics_dataframe(all_results, savepath)

else:

    print("No results available. Run the main analysis loop first.")

    stats_df = None

### Test on Single Subject

Run the analysis on one subject first to verify everything works before the full loop.

In [ ]:
# =============================================================================

# TEST ON SINGLE SUBJECT

# =============================================================================

# Run on one subject first to verify the pipeline works



test_subj = subj_ids[0]

test_roi_source = 'ACC'

test_roi_target = 'HPC'



print(f"Testing spectral GC on: {test_subj}")

print(f"ROI pair: {test_roi_source} -> {test_roi_target}")

band_name = analysis_params.get('band_name', 'frequency band')

print(f"Frequency: {analysis_params['fmin']}-{analysis_params['fmax']} Hz ({band_name})")

print(f"Downsampling to: {analysis_params['target_sfreq']} Hz")

print(f"Max lag: {analysis_params['gc_n_lags'] / analysis_params['target_sfreq'] * 1000:.0f} ms")



# Load epochs with downsampling

test_epochs = load_subject_epochs(

    test_subj, base_dir, analysis_params['event'],

    test_roi_source, test_roi_target,

    tmin=analysis_params['tmin'], tmax=analysis_params['tmax'],

    target_sfreq=analysis_params['target_sfreq'],

    filter_by_power=analysis_params.get('filter_electrodes_by_power', False),

    power_percentile=analysis_params.get('power_percentile_threshold', 50),

    fmin=analysis_params['fmin'],

    fmax=analysis_params['fmax'],

    tfr_time_window=analysis_params.get('tfr_time_window', None),

    use_top_electrode_only=analysis_params.get('use_top_electrode_only', False)

)



if test_epochs is not None:

    print(f"\nLoaded successfully:")

    print(f"  Original sampling rate: {test_epochs.get('original_sfreq', 'N/A')} Hz")

    print(f"  Downsampled to: {test_epochs['sfreq']} Hz")

    print(f"  N epochs: {test_epochs['n_epochs']}")

    print(f"  Time range: {test_epochs['times'][0]:.2f} to {test_epochs['times'][-1]:.2f} s")

    print(f"  N timepoints: {len(test_epochs['times'])}")

    for hemi in ['l', 'r']:

        n_src = test_epochs.get(f'n_source_{hemi}', 0)

        n_tgt = test_epochs.get(f'n_target_{hemi}', 0)

        if n_src > 0 or n_tgt > 0:

            print(f"  Hemisphere {hemi}: {n_src} source, {n_tgt} target channels")

else:

    print("Failed to load epochs!")

In [ ]:
# Compute GC by RPE category for test subject

print("\nComputing spectral GC by RPE category...")



if test_epochs is not None:

    test_results = process_subject_gc(

        test_subj, test_roi_source, test_roi_target, base_dir, learn_df,

        event=analysis_params['event'],

        tmin=analysis_params['tmin'],

        tmax=analysis_params['tmax'],

        target_sfreq=analysis_params['target_sfreq'],

        fmin=analysis_params['fmin'],

        fmax=analysis_params['fmax'],

        gc_n_lags=analysis_params['gc_n_lags'],

        n_freqs=analysis_params.get('n_freqs'),

        verbose=True,

        filter_by_power=analysis_params.get('filter_electrodes_by_power', False),

        power_percentile=analysis_params.get('power_percentile_threshold', 50),

        tfr_time_window=analysis_params.get('tfr_time_window', None),

        use_top_electrode_only=analysis_params.get('use_top_electrode_only', False)

    )

    

    if test_results is not None:

        print("\n" + "="*60)

        print("Test Results Summary:")

        print("="*60)

        

        for hemi, hemi_data in test_results['gc_by_hemi'].items():

            print(f"\nHemisphere {hemi}:")

            print(f"  Source channels: {hemi_data['n_source']}")

            print(f"  Target channels: {hemi_data['n_target']}")

            

            for rpe_cat in ['neg', 'neutral', 'pos']:

                if rpe_cat in hemi_data and hemi_data[rpe_cat]['success']:

                    gc_data = hemi_data[rpe_cat]

                    print(f"  {rpe_cat.upper()}:")

                    print(f"    N trials: {gc_data['n_trials']}")

                    print(f"    Mean forward GC: {np.mean(gc_data['gc_forward']):.4f}")

                    print(f"    Mean backward GC: {np.mean(gc_data['gc_backward']):.4f}")

                    print(f"    Mean net GC: {np.mean(gc_data['gc_net']):.4f}")

    else:

        print("GC computation failed!")

else:

    print("No test epochs available")

In [ ]:
# =============================================================================

# COMPREHENSIVE VISUALIZATION: Frequency Band Power + GC Time Series

# =============================================================================

# Combines hemispheres and shows both underlying frequency band activity and GC dynamics



if test_results is not None:

    # Combine GC results across hemispheres

    combined_gc = combine_hemispheres(test_results['gc_by_hemi'])

    

    # Compute frequency band power for source and target ROIs

    print("Extracting frequency band power for source and target ROIs...")

    band_power_results = compute_band_power_by_rpe(

        test_epochs, learn_df, test_subj,

        fmin=analysis_params['fmin'], 

        fmax=analysis_params['fmax']

    )

    

    # Check which RPE categories have valid data

    rpe_cats = ['neg', 'neutral', 'pos']

    valid_cats = [cat for cat in rpe_cats if cat in combined_gc and combined_gc[cat].get('success', False)]

    

    

    # Align band power times with GC times

    if len(valid_cats) > 0:

        gc_times = combined_gc[valid_cats[0]]['times']

        band_power_times = band_power_results.get('times', None)

        

        # If times don't match, interpolate band power to GC times

        if band_power_times is not None and len(band_power_times) != len(gc_times):

            from scipy.interpolate import interp1d

            print(f"  Warning: Time mismatch - GC: {len(gc_times)}, Band power: {len(band_power_times)}. Interpolating...")

            

            for roi_type in ['source', 'target']:

                for cat in rpe_cats:

                    if band_power_results[roi_type].get(cat) is not None:

                        old_mean = band_power_results[roi_type][cat]['mean']

                        old_sem = band_power_results[roi_type][cat]['sem']

                        

                        # Interpolate

                        interp_mean = interp1d(band_power_times, old_mean, kind='linear', 

                                              bounds_error=False, fill_value='extrapolate')

                        interp_sem = interp1d(band_power_times, old_sem, kind='linear',

                                             bounds_error=False, fill_value='extrapolate')

                        

                        band_power_results[roi_type][cat]['mean'] = interp_mean(gc_times)

                        band_power_results[roi_type][cat]['sem'] = interp_sem(gc_times)

            

            band_power_results['times'] = gc_times



    if len(valid_cats) < 2:

        print(f"Insufficient valid RPE categories: {valid_cats}")

    else:

        # Create 2x3 figure

        fig, axes = plt.subplots(2, 3, figsize=(16, 10))

        

        # Colors and labels for RPE categories

        colors = {'neg': '#3498db', 'neutral': '#95a5a6', 'pos': '#e74c3c'}

        labels = {'neg': 'Negative RPE', 'neutral': 'Neutral RPE', 'pos': 'Positive RPE'}

        

        # Get times from GC (this is the reference)

        times = combined_gc[valid_cats[0]]['times']

        

        # Verify and fix time alignment - ensure band power matches GC times

        band_power_times = band_power_results.get('times', None)

        if band_power_times is not None and len(band_power_times) != len(times):

            from scipy.interpolate import interp1d

            print(f"  Interpolating band power: {len(band_power_times)} -> {len(times)} timepoints")

            

            for roi_type in ['source', 'target']:

                for cat in rpe_cats:

                    if band_power_results[roi_type].get(cat) is not None:

                        old_mean = band_power_results[roi_type][cat]['mean']

                        old_sem = band_power_results[roi_type][cat]['sem']

                        

                        # Only interpolate if lengths don't match

                        if len(old_mean) != len(times):

                            # Interpolate mean and sem

                            interp_mean = interp1d(band_power_times, old_mean, kind='linear', 

                                                  bounds_error=False, fill_value='extrapolate')

                            interp_sem = interp1d(band_power_times, old_sem, kind='linear',

                                                 bounds_error=False, fill_value='extrapolate')

                            

                            band_power_results[roi_type][cat]['mean'] = interp_mean(times)

                            band_power_results[roi_type][cat]['sem'] = interp_sem(times)

            

            # Update times in band_power_results to match

            band_power_results['times'] = times

        

        # ========== ROW 1: FREQUENCY BAND POWER ==========

        

        # Panel 1: Source ROI band power

        ax = axes[0, 0]

        for cat in valid_cats:

            if band_power_results['source'][cat] is not None:

                mean_power = band_power_results['source'][cat]['mean']

                sem = band_power_results['source'][cat]['sem']

                n = band_power_results['source'][cat]['n_trials']

                ax.plot(times, mean_power, color=colors[cat], 

                       label=f"{labels[cat]} (n={n})", linewidth=2)

                ax.fill_between(times, mean_power - sem, mean_power + sem, 

                               color=colors[cat], alpha=0.2)

        ax.axvline(0, color='gray', linestyle='--', alpha=0.5)

        ax.set_xlabel('Time (s)')

        ax.set_ylabel('Frequency Band Power (a.u.)')

        ax.set_title(f'Source: {test_roi_source}')

        ax.legend(fontsize=8, loc='upper right')

        

        # Panel 2: Target ROI band power

        ax = axes[0, 1]

        for cat in valid_cats:

            if band_power_results['target'][cat] is not None:

                mean_power = band_power_results['target'][cat]['mean']

                sem = band_power_results['target'][cat]['sem']

                n = band_power_results['target'][cat]['n_trials']

                ax.plot(times, mean_power, color=colors[cat], 

                       label=f"{labels[cat]} (n={n})", linewidth=2)

                ax.fill_between(times, mean_power - sem, mean_power + sem, 

                               color=colors[cat], alpha=0.2)

        ax.axvline(0, color='gray', linestyle='--', alpha=0.5)

        ax.set_xlabel('Time (s)')

        ax.set_ylabel('Frequency Band Power (a.u.)')

        ax.set_title(f'Target: {test_roi_target}')

        ax.legend(fontsize=8, loc='upper right')

        

        # Panel 3: Both ROIs overlaid (using neutral RPE as example)

        ax = axes[0, 2]

        roi_colors = {'source': '#2ecc71', 'target': '#9b59b6'}

        for roi_type, roi_name in [('source', test_roi_source), ('target', test_roi_target)]:

            if band_power_results[roi_type]['neutral'] is not None:

                mean_power = band_power_results[roi_type]['neutral']['mean']

                sem = band_power_results[roi_type]['neutral']['sem']

                ax.plot(times, mean_power, color=roi_colors[roi_type], 

                       label=f'{roi_name}', linewidth=2)

                ax.fill_between(times, mean_power - sem, mean_power + sem, 

                               color=roi_colors[roi_type], alpha=0.2)

        ax.axvline(0, color='gray', linestyle='--', alpha=0.5)

        ax.set_xlabel('Time (s)')

        ax.set_ylabel('Frequency Band Power (a.u.)')

        ax.set_title('Source vs Target (Neutral RPE)')

        ax.legend(fontsize=8, loc='upper right')

        

        # ========== ROW 2: GRANGER CAUSALITY ==========

        

        # Panel 4: Forward GC (source -> target)

        ax = axes[1, 0]

        for cat in valid_cats:

            gc_data = combined_gc[cat]

            n_hemi = gc_data.get('n_hemispheres', 1)

            label_suffix = f" ({n_hemi}H)" if n_hemi > 1 else ""

            ax.plot(times, gc_data['gc_forward'], color=colors[cat], 

                   label=f"{labels[cat]}{label_suffix}", linewidth=2)

        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

        ax.axvline(0, color='gray', linestyle='--', alpha=0.5)

        ax.set_xlabel('Time (s)')

        ax.set_ylabel('GC')

        ax.set_title(f'Forward GC: {test_roi_source} → {test_roi_target}')

        ax.legend(fontsize=8, loc='upper right')

        

        # Panel 5: Backward GC (target -> source)

        ax = axes[1, 1]

        for cat in valid_cats:

            gc_data = combined_gc[cat]

            ax.plot(times, gc_data['gc_backward'], color=colors[cat], 

                   label=labels[cat], linewidth=2)

        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

        ax.axvline(0, color='gray', linestyle='--', alpha=0.5)

        ax.set_xlabel('Time (s)')

        ax.set_ylabel('GC')

        ax.set_title(f'Backward GC: {test_roi_target} → {test_roi_source}')

        ax.legend(fontsize=8, loc='upper right')

        

        # Panel 6: Net GC (forward - backward)

        ax = axes[1, 2]

        for cat in valid_cats:

            gc_data = combined_gc[cat]

            ax.plot(times, gc_data['gc_net'], color=colors[cat], 

                   label=labels[cat], linewidth=2)

            ax.fill_between(times, 0, gc_data['gc_net'], alpha=0.2, color=colors[cat])

        ax.axhline(0, color='black', linestyle='-', linewidth=1)

        ax.axvline(0, color='gray', linestyle='--', alpha=0.5)

        ax.set_xlabel('Time (s)')

        ax.set_ylabel('Net GC (Fwd - Bwd)')

        ax.set_title(f'Net GC: {test_roi_source} ↔ {test_roi_target}')

        ax.legend(fontsize=8, loc='upper right')

        

        # Overall title

        band_name = analysis_params.get('band_name', 'frequency band')

        fig.suptitle(f'{test_subj}: {test_roi_source} → {test_roi_target} | {band_name.capitalize()} ({analysis_params["fmin"]}-{analysis_params["fmax"]} Hz)', 

                     fontsize=14, fontweight='bold', y=1.02)

        

        plt.tight_layout()

        plt.show()

        

        # Print summary statistics

        print("\n" + "="*60)

        print("COMBINED RESULTS SUMMARY (hemispheres averaged)")

        print("="*60)

        for cat in valid_cats:

            gc_data = combined_gc[cat]

            print(f"\n{cat.upper()}:")

            print(f"  N trials: {gc_data['n_trials']}")

            print(f"  N hemispheres: {gc_data.get('n_hemispheres', 1)}")

            print(f"  Mean Forward GC: {np.mean(gc_data['gc_forward']):.4f}")

            print(f"  Mean Backward GC: {np.mean(gc_data['gc_backward']):.4f}")

            print(f"  Mean Net GC: {np.mean(gc_data['gc_net']):.4f}")

else:

    print("No test results available - run cell 17 first")

In [ ]:
gc_data['gc_net'].shape

In [ ]:
plt.plot(gc_data['times'], gc_data['gc_net'])

In [ ]:
        band_name = analysis_params.get('band_name', 'frequency band')

# Bar plot comparing mean GC across RPE categoriesprint("\nMean GC comparison by RPE category...")if test_results is not None:    for hemi, hemi_data in test_results['gc_by_hemi'].items():        rpe_cats = ['neg', 'neutral', 'pos']        valid_cats = [cat for cat in rpe_cats if cat in hemi_data and hemi_data[cat].get('success', False)]                if len(valid_cats) < 2:            continue                fig, ax = plt.subplots(figsize=(10, 5))                x = np.arange(len(valid_cats))        width = 0.25                fwd_means = [np.mean(hemi_data[cat]['gc_forward']) for cat in valid_cats]        bwd_means = [np.mean(hemi_data[cat]['gc_backward']) for cat in valid_cats]        net_means = [np.mean(hemi_data[cat]['gc_net']) for cat in valid_cats]                ax.bar(x - width, fwd_means, width, label=f'Forward ({test_roi_source}→{test_roi_target})',                color='#2ecc71', edgecolor='black')        ax.bar(x, bwd_means, width, label=f'Backward ({test_roi_target}→{test_roi_source})',                color='#e74c3c', edgecolor='black')        ax.bar(x + width, net_means, width, label='Net GC',                color='#3498db', edgecolor='black')                ax.set_xticks(x)        ax.set_xticklabels([f"{cat.capitalize()}\n(n={hemi_data[cat]['n_trials']})" for cat in valid_cats])        ax.axhline(0, color='black', linewidth=1)        ax.set_xlabel('RPE Category')        ax.set_ylabel('Mean GC')        ax.set_title(f'{test_subj}: {test_roi_source} ↔ {test_roi_target} | Hemisphere {hemi.upper()}\n'                    f'Mean {band_name.capitalize()} GC by RPE Category')        ax.legend(loc='upper right')                plt.tight_layout()        plt.show()            print("\nInterpretation:")    print("  Positive Net GC → ACC dominantly drives target region")    print("  Negative Net GC → Target region dominantly drives ACC")    print("  Differences by RPE category suggest reward-modulated connectivity")

In [ ]:
# Step 4: Comprehensive GC Verification Visualization

# Shows three complementary views demonstrating what Granger causality captures:

# 1. Lagged overlay: Does shifting source align with target?

# 2. Lagged scatter: Is there predictive correlation?

# 3. VAR prediction: Does including source improve target prediction?



print("\n4. Comprehensive GC verification plots...")

print("   GC measures: Does past SOURCE help predict future TARGET beyond target's own history?")



if 'test_df' in dir() and 'test_hfa' in dir() and test_df.success.sum() > 0:

    from scipy import stats

    from statsmodels.tsa.api import VAR

    

    # Get successful trials sorted by z-scored net GC

    valid_trials = test_df[test_df.success].copy()

    

    # Ensure gc_net exists

    if 'gc_net' not in valid_trials.columns:

        valid_trials['gc_net'] = valid_trials['gc_forward'] - valid_trials['gc_backward']

    

    # Use z-scored values if available, otherwise fall back to raw

    if 'gc_net_z' in valid_trials.columns and valid_trials['gc_net_z'].notna().any():

        sort_col = 'gc_net_z'

        print("   Selecting trials based on Z-SCORED net GC (relative to surrogate)")

    else:

        sort_col = 'gc_net'

        print("   Selecting trials based on RAW net GC (z-scores not available)")

    

    # Select high GC trial (most positive) and near-zero GC trial (closest to 0)

    high_gc_row = valid_trials.nlargest(1, sort_col).iloc[0]

    

    # Find trial closest to zero (smallest absolute value)

    valid_trials['abs_gc'] = valid_trials[sort_col].abs()

    low_gc_row = valid_trials.nsmallest(1, 'abs_gc').iloc[0]

    

    high_gc_trial = int(high_gc_row['trial'])

    low_gc_trial = int(low_gc_row['trial'])

    

    # Print with z-score if available

    if 'gc_net_z' in high_gc_row:

        print(f"\n   High net GC trial: {high_gc_trial}")

        print(f"      Raw: Fwd={high_gc_row['gc_forward']:.1f}, Bwd={high_gc_row['gc_backward']:.1f}, Net={high_gc_row['gc_net']:.2f}")

        print(f"      Z-scored: Net_z={high_gc_row['gc_net_z']:.2f}")

        print(f"   Near-zero net GC trial: {low_gc_trial}")

        print(f"      Raw: Fwd={low_gc_row['gc_forward']:.1f}, Bwd={low_gc_row['gc_backward']:.1f}, Net={low_gc_row['gc_net']:.2f}")

        print(f"      Z-scored: Net_z={low_gc_row['gc_net_z']:.2f}")

    else:

        print(f"\n   High net GC trial: {high_gc_trial} (Fwd={high_gc_row['gc_forward']:.1f}, Bwd={high_gc_row['gc_backward']:.1f}, Net={high_gc_row['gc_net']:.2f})")

        print(f"   Near-zero net GC trial: {low_gc_trial} (Fwd={low_gc_row['gc_forward']:.1f}, Bwd={low_gc_row['gc_backward']:.1f}, Net={low_gc_row['gc_net']:.2f})")

    

    # Extract HFA data

    source_hfa = test_hfa[f'source_hfa_{test_hemi}']

    target_hfa = test_hfa[f'target_hfa_{test_hemi}']

    times = test_hfa['times']

    sfreq = test_hfa['sfreq']

    

    # Apply time window

    time_window = analysis_params['time_windows'].get(test_event)

    if time_window is not None:

        time_mask = (times >= time_window[0]) & (times <= time_window[1])

        times_plot = times[time_mask]

    else:

        time_mask = np.ones(len(times), dtype=bool)

        times_plot = times

    

    # Note: Each trial has its own optimal lag selected by AIC during VAR fitting

    # We'll use the trial-specific optimal_lag in the visualizations

    max_lag = analysis_params['max_lag']

    print(f"   Max lag = {max_lag} samples ({max_lag * (1000/sfreq):.0f} ms at {sfreq} Hz)")

    print(f"   (Each trial uses its own AIC-selected optimal lag)")

    

    trials_to_plot = [

        (high_gc_trial, high_gc_row, 'High Net GC', '#2ecc71'),

        (low_gc_trial, low_gc_row, 'Near-Zero Net GC', '#95a5a6')

    ]

    

    # =================================================================

    # FIGURE 1: Comprehensive 2x5 visualization for skeptical reviewers

    # =================================================================

    fig, axes = plt.subplots(2, 5, figsize=(24, 10))

    

    for row_idx, (trial_idx, trial_row, label, color) in enumerate(trials_to_plot):

        # Get the ACTUAL optimal lag used by VAR for this trial

        trial_optimal_lag = int(trial_row['optimal_lag']) if 'optimal_lag' in trial_row and not np.isnan(trial_row['optimal_lag']) else max_lag

        trial_lag_ms = trial_optimal_lag * (1000 / sfreq)

        

        # Get mean HFA for this trial

        source_trial = source_hfa[trial_idx, :, :][:, time_mask]

        target_trial = target_hfa[trial_idx, :, :][:, time_mask]

        source_mean = np.mean(source_trial, axis=0)

        target_mean = np.mean(target_trial, axis=0)

        

        # Standardize for comparison

        source_z = (source_mean - np.mean(source_mean)) / np.std(source_mean)

        target_z = (target_mean - np.mean(target_mean)) / np.std(target_mean)

        

        # Create lagged pairs for BOTH directions using trial-specific optimal lag

        source_past = source_z[:-trial_optimal_lag]

        target_future = target_z[trial_optimal_lag:]

        target_past = target_z[:-trial_optimal_lag]

        source_future = source_z[trial_optimal_lag:]

        

        # Compute correlations for both directions

        slope_fwd, intercept_fwd, r_fwd, p_fwd, _ = stats.linregress(source_past, target_future)

        slope_bwd, intercept_bwd, r_bwd, p_bwd, _ = stats.linregress(target_past, source_future)

        r2_fwd = r_fwd**2

        r2_bwd = r_bwd**2

        

        # -------------------------------------------------------------

        # PANEL 0: Smoothed HFA Time Series with Lagged Overlay

        # -------------------------------------------------------------

        ax = axes[row_idx, 0]

        

        # Shift source forward by trial-specific optimal_lag samples

        source_shifted = np.roll(source_z, trial_optimal_lag)

        source_shifted[:trial_optimal_lag] = np.nan  # Mask wrapped values

        

        # Plot target, shifted source, and original source

        ax.plot(times_plot, target_z, color='#3498db', linewidth=2, 

                label=f'{test_roi_target} (target)', alpha=0.9)

        ax.plot(times_plot, source_shifted, color='#e67e22', linewidth=2, 

                label=f'{test_roi_source} (shifted +{trial_lag_ms:.0f}ms)', alpha=0.9, linestyle='--')

        ax.plot(times_plot, source_z, color='#95a5a6', linewidth=1, 

                label=f'{test_roi_source} (original)', alpha=0.4)

        

        # Compute correlation between shifted source and target

        valid_mask = ~np.isnan(source_shifted)

        if valid_mask.sum() > 10:

            r_shifted, _ = stats.pearsonr(source_shifted[valid_mask], target_z[valid_mask])

            r_original, _ = stats.pearsonr(source_z, target_z)

        else:

            r_shifted, r_original = np.nan, np.nan

        

        ax.set_ylabel('HFA (z-scored)', fontsize=10)

        ax.set_xlabel('Time from Feedback (s)', fontsize=10)

        

        # Show z-scored net GC in title if available

        if 'gc_net_z' in trial_row and not np.isnan(trial_row['gc_net_z']):

            gc_label = f'Net GC={trial_row["gc_net"]:.1f} (z={trial_row["gc_net_z"]:.1f})'

        else:

            gc_label = f'Net GC={trial_row["gc_net"]:.1f}'

        

        ax.set_title(f'{label}: Smoothed HFA (lag={trial_optimal_lag}, {trial_lag_ms:.0f}ms)\n'

                    f'Trial {trial_idx} | {gc_label}\n'

                    f'r(shifted)={r_shifted:.2f}, r(orig)={r_original:.2f}', fontsize=10)

        ax.legend(loc='upper right', fontsize=7)

        ax.axvline(0, color='gray', linestyle='--', alpha=0.3)

        

        # Highlight: better alignment = higher |r(shifted)|

        if abs(r_shifted) > abs(r_original) + 0.05:

            ax.text(0.02, 0.02, 'Shift improves alignment!', transform=ax.transAxes,

                   fontsize=8, va='bottom', color='green', fontweight='bold')

        

        # -------------------------------------------------------------

        # PANEL 1: Forward Scatter (Source → Target)

        # -------------------------------------------------------------

        ax = axes[row_idx, 1]

        

        ax.scatter(source_past, target_future, alpha=0.5, s=25, c='#2ecc71', edgecolors='none')

        x_line = np.array([source_past.min(), source_past.max()])

        y_line = slope_fwd * x_line + intercept_fwd

        ax.plot(x_line, y_line, 'k-', linewidth=2)

        

        ax.set_xlabel(f'{test_roi_source}(t)', fontsize=10)

        ax.set_ylabel(f'{test_roi_target}(t+{trial_lag_ms:.0f}ms)', fontsize=10)

        ax.set_title(f'{label}: FORWARD\n{test_roi_source} → {test_roi_target}\n'

                    f'R²={r2_fwd:.3f}, p={p_fwd:.2e} | lag={trial_optimal_lag}', fontsize=10)

        ax.axhline(0, color='gray', linestyle='-', alpha=0.2)

        ax.axvline(0, color='gray', linestyle='-', alpha=0.2)

        

        # Significance marker

        sig_marker = '***' if p_fwd < 0.001 else '**' if p_fwd < 0.01 else '*' if p_fwd < 0.05 else 'ns'

        ax.text(0.95, 0.95, sig_marker, transform=ax.transAxes, fontsize=14, 

               ha='right', va='top', color='#2ecc71' if p_fwd < 0.05 else 'gray', fontweight='bold')

        

        # -------------------------------------------------------------

        # PANEL 2: Backward Scatter (Target → Source)

        # -------------------------------------------------------------

        ax = axes[row_idx, 2]

        

        ax.scatter(target_past, source_future, alpha=0.5, s=25, c='#e74c3c', edgecolors='none')

        x_line = np.array([target_past.min(), target_past.max()])

        y_line = slope_bwd * x_line + intercept_bwd

        ax.plot(x_line, y_line, 'k-', linewidth=2)

        

        ax.set_xlabel(f'{test_roi_target}(t)', fontsize=10)

        ax.set_ylabel(f'{test_roi_source}(t+{trial_lag_ms:.0f}ms)', fontsize=10)

        ax.set_title(f'{label}: BACKWARD\n{test_roi_target} → {test_roi_source}\n'

                    f'R²={r2_bwd:.3f}, p={p_bwd:.2e} | lag={trial_optimal_lag}', fontsize=10)

        ax.axhline(0, color='gray', linestyle='-', alpha=0.2)

        ax.axvline(0, color='gray', linestyle='-', alpha=0.2)

        

        # Significance marker

        sig_marker = '***' if p_bwd < 0.001 else '**' if p_bwd < 0.01 else '*' if p_bwd < 0.05 else 'ns'

        ax.text(0.95, 0.95, sig_marker, transform=ax.transAxes, fontsize=14,

               ha='right', va='top', color='#e74c3c' if p_bwd < 0.05 else 'gray', fontweight='bold')

        

        # -------------------------------------------------------------

        # PANEL 3: R² Comparison Bar Chart

        # -------------------------------------------------------------

        ax = axes[row_idx, 3]

        

        bar_labels = ['Forward\nR²', 'Backward\nR²']

        bar_values = [r2_fwd, r2_bwd]

        bar_colors = ['#2ecc71', '#e74c3c']

        

        bars = ax.bar(bar_labels, bar_values, color=bar_colors, edgecolor='black', linewidth=1.5)

        

        # Add value labels

        for bar, val in zip(bars, bar_values):

            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,

                   f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

        

        ax.set_ylabel('R² (Lagged Correlation)', fontsize=10)

        ax.set_title(f'{label}: Predictive Power\nWhich direction explains more variance?', fontsize=10)

        ax.set_ylim(0, max(r2_fwd, r2_bwd) * 1.3 + 0.01)

        

        # Winner annotation with arrow

        if r2_fwd > r2_bwd * 1.1:

            ax.annotate('WINNER', xy=(0, r2_fwd), xytext=(0.5, r2_fwd + 0.02),

                       fontsize=10, ha='center', color='#2ecc71', fontweight='bold')

        elif r2_bwd > r2_fwd * 1.1:

            ax.annotate('WINNER', xy=(1, r2_bwd), xytext=(0.5, r2_bwd + 0.02),

                       fontsize=10, ha='center', color='#e74c3c', fontweight='bold')

        

        # -------------------------------------------------------------

        # PANEL 4: GC F-statistic Comparison (Ground Truth)

        # -------------------------------------------------------------

        ax = axes[row_idx, 4]

        

        gc_fwd = trial_row['gc_forward']

        gc_bwd = trial_row['gc_backward']

        gc_net = trial_row['gc_net']

        

        bar_labels = ['Forward\nGC', 'Backward\nGC', 'Net\nGC']

        bar_values = [gc_fwd, gc_bwd, gc_net]

        bar_colors = ['#2ecc71', '#e74c3c', '#3498db' if gc_net > 0 else '#9b59b6']

        

        bars = ax.bar(bar_labels, bar_values, color=bar_colors, edgecolor='black', linewidth=1.5)

        ax.axhline(0, color='k', linestyle='-', linewidth=1)

        

        # Add value labels

        for bar, val in zip(bars, bar_values):

            height = bar.get_height()

            y_pos = height + 0.2 if height >= 0 else height - 0.4

            va = 'bottom' if height >= 0 else 'top'

            ax.text(bar.get_x() + bar.get_width()/2., y_pos,

                   f'{val:.1f}', ha='center', va=va, fontsize=11, fontweight='bold')

        

        ax.set_ylabel('GC (F-statistic)', fontsize=10)

        ax.set_title(f'{label}: Actual GC Values\n(Computed by VAR model)', fontsize=10)

        

        # Final verdict

        if gc_net > 0.5:

            verdict = f'Net GC = +{gc_net:.1f}\n{test_roi_source} → {test_roi_target}'

            ax.text(0.5, 0.02, verdict, transform=ax.transAxes, fontsize=9, 

                   ha='center', color='#2ecc71', fontweight='bold',

                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        elif gc_net < -0.5:

            verdict = f'Net GC = {gc_net:.1f}\n{test_roi_target} → {test_roi_source}'

            ax.text(0.5, 0.02, verdict, transform=ax.transAxes, fontsize=9,

                   ha='center', color='#e74c3c', fontweight='bold',

                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    

    plt.suptitle(f'GC Verification Suite: {test_subj} | {test_roi_source} ↔ {test_roi_target}\n'

                 f'Comparing FORWARD vs BACKWARD predictability to validate Net GC',

                 fontsize=13, y=1.02)

    plt.tight_layout()

    plt.show()

    

    # =================================================================

    # Summary interpretation

    # =================================================================

    print("\n" + "="*85)

    print("   INTERPRETATION GUIDE - How to convince a skeptic:")

    print("="*85)

    print("\n   COLUMN 1 (Smoothed HFA): Raw signals with lagged overlay")

    print("      - Blue: Target signal | Orange dashed: Source shifted forward by lag")

    print("      - Gray: Original source (unshifted)")

    print("      - If shifted source aligns better with target → source leads target")

    print("\n   COLUMN 2 (Forward Scatter): Does Source(t) predict Target(t+lag)?")

    print("      - Green points show lagged relationship")

    print("      - R² and p-value indicate prediction strength")

    print("      - *, **, *** indicate significance levels")

    print("\n   COLUMN 3 (Backward Scatter): Does Target(t) predict Source(t+lag)?")

    print("      - Red points show lagged relationship in OPPOSITE direction")

    print("      - Same analysis, just reversed")

    print("\n   COLUMN 4 (R² Comparison): Which direction has MORE predictive power?")

    print("      - Direct bar comparison of forward vs backward R²")

    print("      - WINNER label shows which direction explains more variance")

    print("      - This should match the GC results!")

    print("\n   COLUMN 5 (Actual GC): The computed Granger Causality F-statistics")

    print("      - Forward GC: How much does source help predict target?")

    print("      - Backward GC: How much does target help predict source?")

    print("      - Net GC = Forward - Backward")

    print("\n   KEY VALIDATION:")

    print("      - R² winner (Col 4) should match GC winner (Col 5)")

    print("      - High Net GC → Forward R² > Backward R² → Source drives Target")

    print("      - Low Net GC → Backward R² > Forward R² → Target drives Source")

    print("="*85)

    

else:

    print("   No data available for visualization")

In [ ]:
test_df_merged

In [ ]:
smf.ols(data=test_df_merged, formula='gc_net_z ~ 1 + C(rpe_cat)').fit().summary()

In [ ]:
def generate_example_plot(subj_id, roi_source, roi_target, mvgc_df, hfa_data, 

                          time_window, sfreq, savepath, show_plot=False):

    """

    Generate and save the comprehensive GC verification plot for a subject/ROI pair.

    

    Parameters

    ----------

    subj_id : str

        Subject ID

    roi_source : str

        Source ROI name

    roi_target : str

        Target ROI name

    mvgc_df : pd.DataFrame

        MVGC results for this subject/ROI pair

    hfa_data : dict

        HFA data from load_subject_hfa()

    time_window : tuple

        (start, end) time window in seconds

    sfreq : float

        Sampling frequency

    savepath : str

        Directory to save plots

    show_plot : bool

        Whether to display the plot (default False for batch processing)

    

    Returns

    -------

    bool : True if plot was generated successfully

    """

    from scipy import stats

    

    # Filter to successful trials

    valid_df = mvgc_df[mvgc_df.success].copy()

    

    if len(valid_df) < 2:

        print(f"      Skipping {subj_id} {roi_source}->{roi_target}: insufficient valid trials")

        return False

    

    # Ensure gc_net and gc_net_z exist

    if 'gc_net' not in valid_df.columns:

        valid_df['gc_net'] = valid_df['gc_forward'] - valid_df['gc_backward']

    

    # Select trials based on z-scored values if available

    if 'gc_net_z' in valid_df.columns and valid_df['gc_net_z'].notna().any():

        sort_col = 'gc_net_z'

    else:

        sort_col = 'gc_net'

    

    # Get high GC and near-zero GC trials

    high_gc_row = valid_df.nlargest(1, sort_col).iloc[0]

    valid_df['abs_gc'] = valid_df[sort_col].abs()

    zero_gc_row = valid_df.nsmallest(1, 'abs_gc').iloc[0]

    

    high_gc_trial = int(high_gc_row['trials'])

    zero_gc_trial = int(zero_gc_row['trials'])

    

    # Convert to 0-indexed for array access (trials are 1-indexed in dataframe)

    high_gc_idx = high_gc_trial - 1

    zero_gc_idx = zero_gc_trial - 1

    

    # Get hemisphere

    hemi = high_gc_row['hemisphere']

    

    # Get HFA data

    source_hfa = hfa_data.get(f'source_hfa_{hemi}')

    target_hfa = hfa_data.get(f'target_hfa_{hemi}')

    times = hfa_data['times']

    

    if source_hfa is None or target_hfa is None:

        return False

    

    # Apply time window

    if time_window is not None:

        time_mask = (times >= time_window[0]) & (times <= time_window[1])

        times_plot = times[time_mask]

    else:

        time_mask = np.ones(len(times), dtype=bool)

        times_plot = times

    

    trials_to_plot = [

        (high_gc_idx, high_gc_trial, high_gc_row, 'High Net GC', '#2ecc71'),

        (zero_gc_idx, zero_gc_trial, zero_gc_row, 'Near-Zero Net GC', '#95a5a6')

    ]

    

    # Create figure

    fig, axes = plt.subplots(2, 5, figsize=(24, 10))

    

    for row_idx, (arr_idx, trial_num, trial_row, label, color) in enumerate(trials_to_plot):

        # Get trial-specific optimal lag

        trial_optimal_lag = int(trial_row['optimal_lag']) if 'optimal_lag' in trial_row and not np.isnan(trial_row['optimal_lag']) else 10

        trial_lag_ms = trial_optimal_lag * (1000 / sfreq)

        

        # Get mean HFA for this trial (arr_idx is 0-indexed, trial_num is 1-indexed for display)

        source_trial = source_hfa[arr_idx, :, :][:, time_mask]

        target_trial = target_hfa[arr_idx, :, :][:, time_mask]

        source_mean = np.mean(source_trial, axis=0)

        target_mean = np.mean(target_trial, axis=0)

        

        # Standardize

        source_z = (source_mean - np.mean(source_mean)) / np.std(source_mean)

        target_z = (target_mean - np.mean(target_mean)) / np.std(target_mean)

        

        # Create lagged pairs

        source_past = source_z[:-trial_optimal_lag]

        target_future = target_z[trial_optimal_lag:]

        target_past = target_z[:-trial_optimal_lag]

        source_future = source_z[trial_optimal_lag:]

        

        # Compute correlations

        slope_fwd, intercept_fwd, r_fwd, p_fwd, _ = stats.linregress(source_past, target_future)

        slope_bwd, intercept_bwd, r_bwd, p_bwd, _ = stats.linregress(target_past, source_future)

        r2_fwd = r_fwd**2

        r2_bwd = r_bwd**2

        

        # PANEL 0: Smoothed HFA with lagged overlay

        ax = axes[row_idx, 0]

        source_shifted = np.roll(source_z, trial_optimal_lag)

        source_shifted[:trial_optimal_lag] = np.nan

        

        ax.plot(times_plot, target_z, color='#3498db', linewidth=2, label=f'{roi_target} (target)', alpha=0.9)

        ax.plot(times_plot, source_shifted, color='#e67e22', linewidth=2, 

                label=f'{roi_source} (shifted +{trial_lag_ms:.0f}ms)', alpha=0.9, linestyle='--')

        ax.plot(times_plot, source_z, color='#95a5a6', linewidth=1, label=f'{roi_source} (orig)', alpha=0.4)

        

        gc_label = f'Net GC={trial_row["gc_net"]:.1f}'

        if 'gc_net_z' in trial_row and not np.isnan(trial_row['gc_net_z']):

            gc_label += f' (z={trial_row["gc_net_z"]:.1f})'

        

        ax.set_title(f'{label}: HFA (lag={trial_optimal_lag})\nTrial {trial_num} | {gc_label}', fontsize=9)

        ax.set_xlabel('Time (s)')

        ax.set_ylabel('HFA (z)')

        ax.legend(fontsize=6, loc='upper right')

        

        # PANEL 1: Forward scatter

        ax = axes[row_idx, 1]

        ax.scatter(source_past, target_future, alpha=0.5, s=20, c='#2ecc71')

        x_line = np.array([source_past.min(), source_past.max()])

        ax.plot(x_line, slope_fwd * x_line + intercept_fwd, 'k-', linewidth=2)

        ax.set_title(f'FORWARD: {roi_source}→{roi_target}\nR²={r2_fwd:.3f}, p={p_fwd:.2e}', fontsize=9)

        ax.set_xlabel(f'{roi_source}(t)')

        ax.set_ylabel(f'{roi_target}(t+{trial_lag_ms:.0f}ms)')

        

        # PANEL 2: Backward scatter

        ax = axes[row_idx, 2]

        ax.scatter(target_past, source_future, alpha=0.5, s=20, c='#e74c3c')

        x_line = np.array([target_past.min(), target_past.max()])

        ax.plot(x_line, slope_bwd * x_line + intercept_bwd, 'k-', linewidth=2)

        ax.set_title(f'BACKWARD: {roi_target}→{roi_source}\nR²={r2_bwd:.3f}, p={p_bwd:.2e}', fontsize=9)

        ax.set_xlabel(f'{roi_target}(t)')

        ax.set_ylabel(f'{roi_source}(t+{trial_lag_ms:.0f}ms)')

        

        # PANEL 3: R² comparison

        ax = axes[row_idx, 3]

        bars = ax.bar(['Forward\nR²', 'Backward\nR²'], [r2_fwd, r2_bwd], color=['#2ecc71', '#e74c3c'])

        for bar, val in zip(bars, [r2_fwd, r2_bwd]):

            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,

                   f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

        ax.set_ylabel('R²')

        ax.set_title('Predictive Power\nComparison', fontsize=9)

        

        # PANEL 4: GC values

        ax = axes[row_idx, 4]

        gc_fwd = trial_row['gc_forward']

        gc_bwd = trial_row['gc_backward']

        gc_net = trial_row['gc_net']

        bar_colors = ['#2ecc71', '#e74c3c', '#3498db' if gc_net > 0 else '#9b59b6']

        bars = ax.bar(['Fwd GC', 'Bwd GC', 'Net GC'], [gc_fwd, gc_bwd, gc_net], color=bar_colors)

        ax.axhline(0, color='k', linewidth=1)

        for bar, val in zip(bars, [gc_fwd, gc_bwd, gc_net]):

            y_pos = val + 0.2 if val >= 0 else val - 0.4

            ax.text(bar.get_x() + bar.get_width()/2., y_pos, f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')

        ax.set_ylabel('GC (F-stat)')

        ax.set_title('Actual GC Values', fontsize=9)

    

    plt.suptitle(f'{subj_id}: {roi_source} → {roi_target} | GC Verification', fontsize=14)

    plt.tight_layout()

    

    # Save plot

    plot_file = f'{plot_dir}/{subj_id}_{roi_source}_to_{roi_target}_example_trials.pdf'

    plt.savefig(plot_file, dpi=150, bbox_inches='tight')

    

    if show_plot:

        plt.show()

    else:

        plt.close()

    

    return True





print("Example plot function defined")

In [ ]:
def generate_summary_plot(subj_id, roi_source, roi_target, mvgc_df, savepath, show_plot=False):

    """

    Generate and save summary distribution plots for a subject/ROI pair.

    

    Parameters

    ----------

    subj_id : str

        Subject ID

    roi_source : str

        Source ROI name

    roi_target : str

        Target ROI name

    mvgc_df : pd.DataFrame

        MVGC results for this subject/ROI pair

    savepath : str

        Directory to save plots

    show_plot : bool

        Whether to display the plot

    

    Returns

    -------

    bool : True if plot was generated successfully

    """

    valid_df = mvgc_df[mvgc_df.success].copy()

    

    if len(valid_df) < 5:

        print(f"      Skipping summary plot {subj_id} {roi_source}->{roi_target}: insufficient trials")

        return False

    

    # Ensure gc_net exists

    if 'gc_net' not in valid_df.columns:

        valid_df['gc_net'] = valid_df['gc_forward'] - valid_df['gc_backward']

    

    # Create 2x2 summary plot

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    

    # Panel 1: Z-scored GC distribution (if available)

    ax = axes[0, 0]

    if 'gc_net_z' in valid_df.columns and valid_df['gc_net_z'].notna().any():

        ax.hist(valid_df['gc_net_z'].dropna(), bins=20, color='#3498db', edgecolor='black', alpha=0.7)

        ax.axvline(0, color='gray', linestyle='--', linewidth=2, label='Null (z=0)')

        ax.axvline(1.96, color='red', linestyle='--', alpha=0.5, label='p<0.05')

        ax.axvline(-1.96, color='red', linestyle='--', alpha=0.5)

        median_z = valid_df['gc_net_z'].median()

        ax.axvline(median_z, color='#e74c3c', linewidth=2, label=f'Median z={median_z:.2f}')

        ax.set_xlabel('Net GC (z-scored)')

        ax.set_title(f'Z-scored Net GC Distribution\n(n={len(valid_df)} trials)')

        ax.legend()

    else:

        ax.hist(valid_df['gc_net'], bins=20, color='#3498db', edgecolor='black', alpha=0.7)

        ax.axvline(0, color='gray', linestyle='--', linewidth=2)

        ax.set_xlabel('Net GC (F-statistic)')

        ax.set_title('Net GC Distribution')

    

    # Panel 2: Forward vs Backward scatter

    ax = axes[0, 1]

    ax.scatter(valid_df['gc_forward'], valid_df['gc_backward'], alpha=0.6, c='#9b59b6')

    max_val = max(valid_df['gc_forward'].max(), valid_df['gc_backward'].max())

    ax.plot([0, max_val], [0, max_val], 'k--', linewidth=1, label='Unity line')

    ax.set_xlabel(f'Forward GC ({roi_source}→{roi_target})')

    ax.set_ylabel(f'Backward GC ({roi_target}→{roi_source})')

    ax.set_title('Forward vs Backward GC')

    ax.legend()

    

    # Panel 3: Surrogate comparison (if available)

    ax = axes[1, 0]

    if 'surr_mean_net' in valid_df.columns:

        real_mean = valid_df['gc_net'].mean()

        surr_mean = valid_df['surr_mean_net'].iloc[0]  # Same for all trials

        surr_95 = valid_df['surr_95_net'].iloc[0] if 'surr_95_net' in valid_df.columns else None

        

        ax.bar(['Real\n(mean)', 'Surrogate\n(mean)'], [real_mean, surr_mean], 

               color=['#2ecc71', '#95a5a6'], edgecolor='black')

        if surr_95 is not None:

            ax.axhline(surr_95, color='red', linestyle='--', label=f'Surr 95th: {surr_95:.2f}')

        ax.axhline(0, color='black', linewidth=1)

        ax.set_ylabel('Net GC (F-statistic)')

        ax.set_title('Real vs Surrogate Net GC')

        ax.legend()

    else:

        ax.text(0.5, 0.5, 'No surrogate data', transform=ax.transAxes, ha='center')

        ax.set_title('Surrogate Comparison (N/A)')

    

    # Panel 4: Net GC by trial index (temporal pattern)

    ax = axes[1, 1]

    if 'trials' in valid_df.columns:

        valid_df_sorted = valid_df.sort_values('trials')

        ax.plot(valid_df_sorted['trials'], valid_df_sorted['gc_net'], 'o-', alpha=0.6, markersize=4)

        ax.axhline(0, color='gray', linestyle='--')

        ax.set_xlabel('Trial Index')

        ax.set_ylabel('Net GC (F-statistic)')

        ax.set_title('Net GC Over Trials')

    

    plt.suptitle(f'{subj_id}: {roi_source} → {roi_target} | Summary', fontsize=14)

    plt.tight_layout()

    

    # Save plot

    plot_file = f'{plot_dir}/{subj_id}_{roi_source}_to_{roi_target}_summary.pdf'

    plt.savefig(plot_file, dpi=150, bbox_inches='tight')

    

    if show_plot:

        plt.show()

    else:

        plt.close()

    

    return True





print("Summary plot function defined")

---
## Main Analysis Loop

**Run the cells below only after verifying the test results above look reasonable.**

In [ ]:
%%capture --no-display



# Run main analysis loop

# Process each subject individually to save per-subject results and generate plots



import os



# Create output directories

plots_dir = f'{savepath}/plots'

results_dir = f'{savepath}/per_subject'

os.makedirs(plots_dir, exist_ok=True)

os.makedirs(results_dir, exist_ok=True)



all_results = {}



for roi_source, roi_target in analysis_params['roi_pairs']:

    pair_key = f'{roi_source}_{roi_target}'

    print(f"\n{'='*60}")

    print(f"Processing ROI pair: {roi_source} -> {roi_target}")

    print(f"{'='*60}")

    

    for event in analysis_params['events']:

        print(f"\n  Event: {event}")

        

        time_window = analysis_params['time_windows'].get(event)

        result_key = f'{pair_key}_{event}'

        pair_results = []

        

        for subj_id in tqdm(subj_ids, desc=f'{roi_source}->{roi_target}'):

            # Process subject

            subj_results = process_subject_mvgc(

                subj_id, roi_source, roi_target, event, base_dir,

                time_window,

                analysis_params['hfa_freq_range'],

                analysis_params['smooth_window_ms'],

                analysis_params['target_sfreq'],

                analysis_params['max_lag'],

                analysis_params['n_surrogates']

            )

            

            if len(subj_results) == 0:

                continue

            

            # Combine hemispheres if multiple

            subj_df = pd.concat(subj_results, ignore_index=True)

            pair_results.append(subj_df)

            

            # Save per-subject dataframe

            subj_save_file = f'{results_dir}/{subj_id}_{result_key}_mvgc.csv'

            subj_df.to_csv(subj_save_file, index=False)

            

            # Generate plots if there are successful trials

            if subj_df.success.sum() >= 2:

                # Load HFA data for plotting

                hfa_data = load_subject_hfa(

                    subj_id, base_dir, event, roi_source, roi_target,

                    hfa_freq_range=analysis_params['hfa_freq_range'],

                    smooth_window_ms=analysis_params['smooth_window_ms'],

                    target_sfreq=analysis_params['target_sfreq']

                )

                

                if hfa_data is not None:

                    # Generate summary plot

                    generate_summary_plot(

                        subj_id, roi_source, roi_target, subj_df, 

                        plots_dir, show_plot=False

                    )

                    

                    # Generate example trials plot

                    sfreq = hfa_data.get('sfreq', analysis_params['target_sfreq'])

                    generate_example_plot(

                        subj_id, roi_source, roi_target, subj_df, hfa_data,

                        time_window, sfreq, plots_dir, show_plot=False

                    )

        

        # Combine all subjects for this ROI pair

        if len(pair_results) > 0:

            result_df = pd.concat(pair_results, ignore_index=True)

            all_results[result_key] = result_df

            

            # Save combined results

            save_file = f'{savepath}/{result_key}_mvgc_raw.csv'

            result_df.to_csv(save_file, index=False)

            

            # Print summary

            n_subj = result_df.participant.nunique()

            n_trials = len(result_df)

            success_rate = result_df.success.mean() * 100

            print(f"    Combined file: {save_file}")

            print(f"    Subjects: {n_subj}, Trials: {n_trials}, Success rate: {success_rate:.1f}%")

        else:

            print(f"    No valid data for {roi_source}->{roi_target} {event}")



print(f"\n{'='*60}")

print("Analysis complete!")

print(f"Results saved to: {savepath}")

print(f"Per-subject results: {results_dir}")

print(f"Plots: {plots_dir}")

print(f"{'='*60}")

In [ ]:
# Aggregate Results Visualization by Target ROI

# Creates summary plots comparing GC across all target regions



import seaborn as sns



# Define target ROIs (source is ACC)

target_rois = ['dmPFC', 'OFC', 'dlPFC', 'HPC', 'AMY']

source_roi = 'ACC'

event = 'feedback_start'



# Collect data for each target ROI

aggregate_data = []

for target in target_rois:

    result_key = f'{source_roi}_{target}_{event}'

    if result_key in all_results:

        df = all_results[result_key].copy()

        df['target_roi'] = target

        aggregate_data.append(df)



if len(aggregate_data) > 0:

    agg_df = pd.concat(aggregate_data, ignore_index=True)

    

    # Filter to successful trials only

    agg_valid = agg_df[agg_df.success].copy()

    

    # Ensure gc_net exists

    if 'gc_net' not in agg_valid.columns:

        agg_valid['gc_net'] = agg_valid['gc_forward'] - agg_valid['gc_backward']

    

    print(f"Aggregate data: {len(agg_valid)} trials across {agg_valid.participant.nunique()} subjects")

    print(f"Target ROIs represented: {agg_valid.target_roi.unique().tolist()}")

    

    # Create 2x2 figure

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    

    # =======================================================================

    # Panel 1: Net GC Distribution by Target ROI (Violin/Box)

    # =======================================================================

    ax = axes[0, 0]

    

    # Use z-scored values if available

    if 'gc_net_z' in agg_valid.columns and agg_valid['gc_net_z'].notna().any():

        plot_col = 'gc_net_z'

        ylabel = 'Net GC (z-scored)'

        # Add significance threshold lines

        ax.axhline(1.96, color='red', linestyle='--', alpha=0.5, label='p<0.05 threshold')

        ax.axhline(-1.96, color='red', linestyle='--', alpha=0.5)

        ax.axhline(0, color='gray', linestyle='-', alpha=0.3)

    else:

        plot_col = 'gc_net'

        ylabel = 'Net GC (F-statistic)'

        ax.axhline(0, color='gray', linestyle='-', alpha=0.3)

    

    # Order by median net GC

    roi_order = agg_valid.groupby('target_roi')[plot_col].median().sort_values(ascending=False).index.tolist()

    

    sns.violinplot(data=agg_valid, x='target_roi', y=plot_col, order=roi_order, 

                   palette='viridis', ax=ax, inner='box')

    

    # Add significance markers

    for i, roi in enumerate(roi_order):

        roi_data = agg_valid[agg_valid.target_roi == roi][plot_col]

        median_val = roi_data.median()

        if plot_col == 'gc_net_z' and abs(median_val) > 1.96:

            ax.text(i, roi_data.max() + 0.2, '*', fontsize=16, ha='center', fontweight='bold')

    

    ax.set_xlabel('Target ROI', fontsize=12)

    ax.set_ylabel(ylabel, fontsize=12)

    ax.set_title(f'Net GC Distribution by Target ROI\n(Source: {source_roi})', fontsize=12)

    if plot_col == 'gc_net_z':

        ax.legend(loc='upper right', fontsize=9)

    

    # =======================================================================

    # Panel 2: Forward vs Backward GC (Grouped Bar)

    # =======================================================================

    ax = axes[0, 1]

    

    # Compute means per ROI

    gc_means = agg_valid.groupby('target_roi').agg({

        'gc_forward': 'mean',

        'gc_backward': 'mean'

    }).reindex(roi_order)

    

    x = np.arange(len(roi_order))

    width = 0.35

    

    bars1 = ax.bar(x - width/2, gc_means['gc_forward'], width, label='Forward (ACC→Target)', 

                   color='#2ecc71', edgecolor='black')

    bars2 = ax.bar(x + width/2, gc_means['gc_backward'], width, label='Backward (Target→ACC)', 

                   color='#e74c3c', edgecolor='black')

    

    ax.set_xticks(x)

    ax.set_xticklabels(roi_order)

    ax.set_xlabel('Target ROI', fontsize=12)

    ax.set_ylabel('Mean GC (F-statistic)', fontsize=12)

    ax.set_title('Forward vs Backward GC by Target ROI', fontsize=12)

    ax.legend(loc='upper right')

    ax.axhline(0, color='gray', linestyle='-', alpha=0.3)

    

    # Add value labels

    for bar in bars1:

        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,

               f'{bar.get_height():.1f}', ha='center', fontsize=9)

    for bar in bars2:

        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,

               f'{bar.get_height():.1f}', ha='center', fontsize=9)

    

    # =======================================================================

    # Panel 3: Proportion of Significant Trials

    # =======================================================================

    ax = axes[1, 0]

    

    if 'gc_net_z' in agg_valid.columns and agg_valid['gc_net_z'].notna().any():

        # Compute proportion significant (|z| > 1.96)

        prop_sig = []

        n_trials = []

        for roi in roi_order:

            roi_data = agg_valid[agg_valid.target_roi == roi]

            n_sig = (roi_data['gc_net_z'].abs() > 1.96).sum()

            n_total = len(roi_data)

            prop_sig.append(n_sig / n_total if n_total > 0 else 0)

            n_trials.append(n_total)

        

        bars = ax.bar(roi_order, prop_sig, color='#3498db', edgecolor='black')

        ax.axhline(0.05, color='red', linestyle='--', alpha=0.7, label='Expected by chance (5%)')

        

        # Add n labels

        for i, (bar, n) in enumerate(zip(bars, n_trials)):

            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,

                   f'{prop_sig[i]*100:.1f}%\n(n={n})', ha='center', fontsize=9)

        

        ax.set_xlabel('Target ROI', fontsize=12)

        ax.set_ylabel('Proportion Significant', fontsize=12)

        ax.set_title('Proportion of Trials with Significant Net GC\n(|z| > 1.96, two-tailed)', fontsize=12)

        ax.set_ylim(0, max(prop_sig) * 1.3 + 0.05)

        ax.legend(loc='upper right')

    else:

        ax.text(0.5, 0.5, 'Z-scores not available\n(no surrogate analysis)', 

               transform=ax.transAxes, ha='center', va='center', fontsize=12)

        ax.set_title('Proportion Significant (N/A)', fontsize=12)

    

    # =======================================================================

    # Panel 4: Real vs Surrogate Comparison

    # =======================================================================

    ax = axes[1, 1]

    

    if 'surr_mean_net' in agg_valid.columns:

        # Compute means per ROI

        real_means = agg_valid.groupby('target_roi')['gc_net'].mean().reindex(roi_order)

        surr_means = agg_valid.groupby('target_roi')['surr_mean_net'].mean().reindex(roi_order)

        

        x = np.arange(len(roi_order))

        width = 0.35

        

        bars1 = ax.bar(x - width/2, real_means, width, label='Real', 

                       color='#2ecc71', edgecolor='black')

        bars2 = ax.bar(x + width/2, surr_means, width, label='Surrogate', 

                       color='#95a5a6', edgecolor='black')

        

        ax.set_xticks(x)

        ax.set_xticklabels(roi_order)

        ax.set_xlabel('Target ROI', fontsize=12)

        ax.set_ylabel('Mean Net GC (F-statistic)', fontsize=12)

        ax.set_title('Real vs Surrogate Net GC by Target ROI', fontsize=12)

        ax.legend(loc='upper right')

        ax.axhline(0, color='gray', linestyle='-', alpha=0.3)

    else:

        ax.text(0.5, 0.5, 'Surrogate data not available', 

               transform=ax.transAxes, ha='center', va='center', fontsize=12)

        ax.set_title('Real vs Surrogate (N/A)', fontsize=12)

    

    plt.suptitle(f'Aggregate Granger Causality Results\nSource: {source_roi} → Targets | Encoding Period',

                 fontsize=14, y=1.02)

    plt.tight_layout()

    

    # Save figure

    agg_plot_file = f'{plot_dir}/aggregate_{source_roi}_to_targets_summary.pdf'

    plt.savefig(agg_plot_file, dpi=150, bbox_inches='tight')

    plt.show()

    

    print(f"\nAggregate plot saved to: {agg_plot_file}")

    

    # Print summary statistics

    print("\n" + "="*60)

    print("Summary Statistics by Target ROI:")

    print("="*60)

    for roi in roi_order:

        roi_data = agg_valid[agg_valid.target_roi == roi]

        print(f"\n{source_roi} → {roi}:")

        print(f"  N subjects: {roi_data.participant.nunique()}")

        print(f"  N trials: {len(roi_data)}")

        print(f"  Mean Forward GC: {roi_data.gc_forward.mean():.2f}")

        print(f"  Mean Backward GC: {roi_data.gc_backward.mean():.2f}")

        print(f"  Mean Net GC: {roi_data.gc_net.mean():.2f}")

        if 'gc_net_z' in roi_data.columns:

            print(f"  Mean Net GC (z): {roi_data.gc_net_z.mean():.2f}")

            prop_sig = (roi_data.gc_net_z.abs() > 1.96).mean() * 100

            print(f"  % Significant: {prop_sig:.1f}%")

else:

    print("No aggregate data available. Run the main analysis loop first.")

In [ ]:
# Mixed Effects Model: Net GC (z-scored) ~ RPE Category

# For each target ROI separately



import statsmodels.formula.api as smf

from statsmodels.stats.multitest import multipletests



# Define target ROIs

target_rois = ['dmPFC', 'OFC', 'dlPFC', 'HPC', 'AMY']

source_roi = 'ACC'

event = 'feedback_start'



# Collect and merge data for each target ROI

merged_data = []

for target in target_rois:

    result_key = f'{source_roi}_{target}_{event}'

    if result_key in all_results:

        df = all_results[result_key].copy()

        df['target_roi'] = target

        

        # Merge with behavioral data

        merged_df = merge_with_behavior(df, learn_df)

        merged_data.append(merged_df)



if len(merged_data) > 0:

    all_merged = pd.concat(merged_data, ignore_index=True)

    

    # Filter to successful trials with valid data

    model_df = all_merged[

        (all_merged.success) & 

        (all_merged.gc_net_z.notna()) & 

        (all_merged.rpe_cat.notna())

    ].copy()

    

    print(f"Data for mixed effects model:")

    print(f"  Total trials: {len(model_df)}")

    print(f"  Subjects: {model_df.participant.nunique()}")

    print(f"  Target ROIs: {model_df.target_roi.unique().tolist()}")

    print(f"  RPE categories: {model_df.rpe_cat.unique().tolist()}")

    

    # Convert rpe_cat to categorical with neutral as reference

    model_df['rpe_cat'] = pd.Categorical(

        model_df['rpe_cat'], 

        categories=['neutral', 'neg', 'pos'],  # neutral as reference

        ordered=False

    )

    

    # Run mixed effects model for each target ROI

    print("\n" + "="*70)

    print("Mixed Effects Models: gc_net_z ~ rpe_cat + (1|participant)")

    print("Reference category: neutral")

    print("="*70)

    

    model_results = {}

    all_pvals = []

    

    for target in target_rois:

        target_data = model_df[model_df.target_roi == target]

        

        if len(target_data) < 20:

            print(f"\n{source_roi} → {target}: Insufficient data (n={len(target_data)})")

            continue

        

        n_subj = target_data.participant.nunique()

        

        try:

            # Fit mixed effects model with random intercept for participant

            model = smf.mixedlm(

                "gc_net_z ~ C(rpe_cat, Treatment('neutral'))", 

                data=target_data,

                groups=target_data['participant']

            )

            result = model.fit(method='lbfgs', maxiter=500)

            

            model_results[target] = result

            

            print(f"\n{'-'*70}")

            print(f"{source_roi} → {target}")

            print(f"N = {len(target_data)} trials, {n_subj} subjects")

            print(f"{'-'*70}")

            

            # Extract key results

            params = result.params

            pvals = result.pvalues

            conf_int = result.conf_int()

            

            # Print results

            print(f"\n{'Parameter':<35} {'Coef':>10} {'95% CI':>20} {'p-value':>12}")

            print("-"*80)

            

            for param in params.index:

                coef = params[param]

                pval = pvals[param]

                ci_low, ci_high = conf_int.loc[param]

                

                # Significance marker

                sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''

                

                # Clean up parameter name for display

                display_name = param.replace("C(rpe_cat, Treatment('neutral'))", "rpe_cat")

                display_name = display_name.replace("[T.", "[").replace("]", "]")

                

                print(f"{display_name:<35} {coef:>10.3f} [{ci_low:>7.3f}, {ci_high:>7.3f}] {pval:>10.4f} {sig}")

                

                # Collect p-values for multiple comparison correction (excluding intercept and group var)

                if 'Intercept' not in param and 'Group Var' not in param:

                    all_pvals.append({

                        'target': target,

                        'param': param,

                        'pval': pval,

                        'coef': coef

                    })

            

            # Model fit statistics

            print(f"\nLog-likelihood: {result.llf:.2f}")

            print(f"Converged: {result.converged}")

            

        except Exception as e:

            print(f"\n{source_roi} → {target}: Model fitting failed - {str(e)[:50]}")

    

    # Multiple comparison correction across all tests

    if len(all_pvals) > 0:

        pval_df = pd.DataFrame(all_pvals)

        

        # FDR correction

        _, pvals_corrected, _, _ = multipletests(pval_df['pval'], method='fdr_bh')

        pval_df['pval_fdr'] = pvals_corrected

        

        print("\n" + "="*70)

        print("Summary: Significant Effects (FDR-corrected)")

        print("="*70)

        

        sig_effects = pval_df[pval_df.pval_fdr < 0.05]

        if len(sig_effects) > 0:

            print("\nSignificant after FDR correction (q < 0.05):")

            for _, row in sig_effects.iterrows():

                direction = "↑" if row['coef'] > 0 else "↓"

                print(f"  {source_roi} → {row['target']}: {row['param']} (coef={row['coef']:.3f}, q={row['pval_fdr']:.4f}) {direction}")

        else:

            print("\nNo significant effects after FDR correction.")

        

        # Create summary visualization

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        

        # Panel 1: Effect sizes (coefficients) for neg and pos RPE

        ax = axes[0]

        

        # Extract coefficients for each target

        coef_data = []

        for target in target_rois:

            if target in model_results:

                result = model_results[target]

                for param in result.params.index:

                    if 'neg' in param:

                        coef_data.append({'target': target, 'rpe_cat': 'Negative', 

                                         'coef': result.params[param], 'se': result.bse[param]})

                    elif 'pos' in param:

                        coef_data.append({'target': target, 'rpe_cat': 'Positive',

                                         'coef': result.params[param], 'se': result.bse[param]})

        

        if coef_data:

            coef_df = pd.DataFrame(coef_data)

            

            x = np.arange(len([t for t in target_rois if t in model_results]))

            width = 0.35

            targets_with_results = [t for t in target_rois if t in model_results]

            

            neg_coefs = coef_df[coef_df.rpe_cat == 'Negative'].set_index('target').reindex(targets_with_results)

            pos_coefs = coef_df[coef_df.rpe_cat == 'Positive'].set_index('target').reindex(targets_with_results)

            

            bars1 = ax.bar(x - width/2, neg_coefs['coef'], width, yerr=neg_coefs['se']*1.96,

                          label='Negative RPE', color='#3498db', capsize=3)

            bars2 = ax.bar(x + width/2, pos_coefs['coef'], width, yerr=pos_coefs['se']*1.96,

                          label='Positive RPE', color='#e74c3c', capsize=3)

            

            ax.set_xticks(x)

            ax.set_xticklabels(targets_with_results)

            ax.axhline(0, color='gray', linestyle='-', linewidth=1)

            ax.set_xlabel('Target ROI', fontsize=12)

            ax.set_ylabel('Effect on Net GC (z)\nvs. Neutral RPE', fontsize=12)

            ax.set_title('RPE Category Effects on Net GC\n(Mixed Effects Model Coefficients ± 95% CI)', fontsize=12)

            ax.legend(loc='upper right')

        

        # Panel 2: Raw means by RPE category for each ROI

        ax = axes[1]

        

        # Compute means and SEM

        means = model_df.groupby(['target_roi', 'rpe_cat'])['gc_net_z'].agg(['mean', 'sem']).reset_index()

        means = means.pivot(index='target_roi', columns='rpe_cat', values=['mean', 'sem'])

        means = means.reindex([t for t in target_rois if t in means.index])

        

        x = np.arange(len(means))

        width = 0.25

        

        for i, cat in enumerate(['neg', 'neutral', 'pos']):

            if cat in means['mean'].columns:

                color = '#3498db' if cat == 'neg' else '#95a5a6' if cat == 'neutral' else '#e74c3c'

                label = 'Negative' if cat == 'neg' else 'Neutral' if cat == 'neutral' else 'Positive'

                ax.bar(x + (i-1)*width, means['mean'][cat], width, 

                      yerr=means['sem'][cat]*1.96, label=label, color=color, capsize=2)

        

        ax.set_xticks(x)

        ax.set_xticklabels(means.index)

        ax.axhline(0, color='gray', linestyle='-', linewidth=1)

        ax.set_xlabel('Target ROI', fontsize=12)

        ax.set_ylabel('Mean Net GC (z-scored)', fontsize=12)

        ax.set_title('Net GC by RPE Category\n(Mean ± 95% CI)', fontsize=12)

        ax.legend(loc='upper right')

        

        plt.suptitle(f'Mixed Effects Analysis: {source_roi} → Target ROIs\nNet GC ~ RPE Category',

                    fontsize=14, y=1.02)

        plt.tight_layout()

        

        # Save figure

        model_plot_file = f'{plot_dir}/mixed_model_gc_rpe_{source_roi}_to_targets.pdf'

        plt.savefig(model_plot_file, dpi=150, bbox_inches='tight')

        plt.show()

        

        print(f"\nModel plot saved to: {model_plot_file}")

else:

    print("No data available. Run the main analysis loop first.")

In [ ]:
savepath

## Merge with Behavioral Data

In [ ]:
def merge_with_behavior(mvgc_df, learn_df):

    """

    Merge MVGC results with behavioral data from learn_df.

    

    Parameters

    ----------

    mvgc_df : pd.DataFrame

        MVGC results DataFrame

    learn_df : pd.DataFrame

        Learning phase behavioral data

    

    Returns

    -------

    pd.DataFrame : Merged DataFrame with behavioral variables

    """

    # Merge with learning phase data

    behavior_cols = ['participant', 'trials', 'rpe', 'zrpe', 'rpe_cat', 

                    'reward', 'gamble', 'gamble_rt']

    merge_cols = ['participant', 'trials']

    

    merged_df = mvgc_df.merge(

        learn_df[behavior_cols], 

        on=merge_cols, 

        how='left'

    )

    

    return merged_df





# Merge all results with behavior

merged_results = {}



for result_key, result_df in all_results.items():

    merged_df = merge_with_behavior(result_df, learn_df)

    merged_results[result_key] = merged_df

    

    # Save merged results

    save_file = f'{savepath}/{result_key}_mvgc_merged.csv'

    merged_df.to_csv(save_file, index=False)

    

    print(f"Merged {result_key}: {len(merged_df)} rows")

    print(f"  Saved to: {save_file}")



print("\nBehavioral merge complete!")

## Statistical Analysis

In [ ]:
# Load results if not in memory (for rerunning analysis section)

def load_results_if_needed():

    """Load saved results if not already in memory."""

    global merged_results

    if 'merged_results' not in dir() or len(merged_results) == 0:

        merged_results = {}

        for roi_source, roi_target in analysis_params['roi_pairs']:

            pair_key = f'{roi_source}_{roi_target}'

            for event in analysis_params['events']:

                result_key = f'{pair_key}_{event}'

                load_file = f'{savepath}/{result_key}_mvgc_merged.csv'

                if os.path.exists(load_file):

                    merged_results[result_key] = pd.read_csv(load_file)

                    print(f"Loaded: {load_file}")

    return merged_results



merged_results = load_results_if_needed()

In [ ]:
# (Cell cleared - encoding mixed-effects models removed, will add statistical analysis later)

In [ ]:
# (Cell cleared - retrieval mixed-effects models removed)

## Visualization

In [ ]:
# Visualization: Encoding period - Z-scored GC distributions and by RPE category

fig, axes = plt.subplots(2, 4, figsize=(16, 8))



for idx, (roi_source, roi_target) in enumerate(analysis_params['roi_pairs']):

    result_key = f'{roi_source}_{roi_target}_feedback_start'

    

    if result_key not in merged_results:

        continue

    

    df = merged_results[result_key].copy()

    df = df[df.success == True]

    

    # Top row: Distribution of Z-scored GC values

    ax = axes[0, idx]

    if 'gc_forward_z' in df.columns and df['gc_forward_z'].notna().any():

        ax.hist(df['gc_forward_z'].dropna(), bins=20, alpha=0.5, label='Forward', color='blue')

        ax.hist(df['gc_backward_z'].dropna(), bins=20, alpha=0.5, label='Backward', color='red')

        ax.axvline(df['gc_forward_z'].mean(), color='blue', linestyle='--', alpha=0.8)

        ax.axvline(df['gc_backward_z'].mean(), color='red', linestyle='--', alpha=0.8)

        ax.axvline(0, color='k', linestyle='-', alpha=0.3, label='Surrogate mean')

        ax.set_xlabel('Z-scored GC')

    else:

        ax.hist(df['gc_forward'], bins=20, alpha=0.5, label='Forward', color='blue')

        ax.hist(df['gc_backward'], bins=20, alpha=0.5, label='Backward', color='red')

        ax.set_xlabel('GC (F-statistic)')

    ax.set_title(f'{roi_source} → {roi_target}')

    ax.set_ylabel('Count')

    ax.legend(fontsize=8)

    

    # Bottom row: Z-scored Net GC by RPE category

    ax = axes[1, idx]

    df_rpe = df[df.rpe_cat.notna()]

    if len(df_rpe) > 0:

        y_col = 'gc_net_z' if 'gc_net_z' in df_rpe.columns and df_rpe['gc_net_z'].notna().any() else 'gc_net'

        if y_col == 'gc_net' and 'gc_net' not in df_rpe.columns:

            df_rpe['gc_net'] = df_rpe['gc_forward'] - df_rpe['gc_backward']

        sns.barplot(data=df_rpe, x='rpe_cat', y=y_col, 

                    order=['neg', 'neutral', 'pos'],

                    errorbar='se', ax=ax, palette='coolwarm')

        ax.axhline(0, color='k', linestyle='--', alpha=0.5)

    ax.set_xlabel('RPE Category')

    ylabel = 'Net GC (Z-scored)' if 'gc_net_z' in df.columns else 'Net GC (F-stat)'

    ax.set_ylabel(ylabel)



plt.suptitle('Encoding Period: Multivariate Granger Causality (Z-scored)', fontsize=14)

plt.tight_layout()

plt.savefig(f'{plot_dir}/encoding_mvgc_summary.pdf', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# (Cell cleared - retrieval period analysis removed)

In [ ]:
# Visualization: Z-scored GC summary and significance

fig, axes = plt.subplots(2, 4, figsize=(16, 8))



for idx, (roi_source, roi_target) in enumerate(analysis_params['roi_pairs']):

    result_key = f'{roi_source}_{roi_target}_feedback_start'

    

    if result_key not in merged_results:

        continue

    

    df = merged_results[result_key].copy()

    df = df[df.success == True]

    

    if len(df) == 0:

        continue

    

    # Get surrogate stats (same for all trials within subject/hemi)

    subj_stats = df.groupby(['participant', 'hemisphere']).first().reset_index()

    

    # Top row: Z-scored Forward GC distribution

    ax = axes[0, idx]

    if 'gc_forward_z' in df.columns and df['gc_forward_z'].notna().any():

        mean_z_fwd = df['gc_forward_z'].mean()

        ax.hist(df['gc_forward_z'].dropna(), bins=25, alpha=0.7, color='#3498db', edgecolor='white')

        ax.axvline(0, color='k', linestyle='-', linewidth=2, label='Null (z=0)')

        ax.axvline(mean_z_fwd, color='#e74c3c', linestyle='--', linewidth=2, label=f'Mean z={mean_z_fwd:.2f}')

        ax.axvline(1.96, color='gray', linestyle=':', alpha=0.5)

        ax.axvline(-1.96, color='gray', linestyle=':', alpha=0.5, label='±1.96 (p=0.05)')

        ax.set_xlabel('Z-scored Forward GC')

        ax.set_ylabel('Count')

        ax.legend(fontsize=7, loc='upper right')

    else:

        ax.text(0.5, 0.5, 'No z-scored data', ha='center', va='center', transform=ax.transAxes)

    ax.set_title(f'{roi_source} → {roi_target}')

    

    # Bottom row: Z-scored Net GC distribution

    ax = axes[1, idx]

    if 'gc_net_z' in df.columns and df['gc_net_z'].notna().any():

        mean_z_net = df['gc_net_z'].mean()

        ax.hist(df['gc_net_z'].dropna(), bins=25, alpha=0.7, color='#9b59b6', edgecolor='white')

        ax.axvline(0, color='k', linestyle='-', linewidth=2, label='Null (z=0)')

        ax.axvline(mean_z_net, color='#e74c3c', linestyle='--', linewidth=2, label=f'Mean z={mean_z_net:.2f}')

        ax.axvline(1.96, color='gray', linestyle=':', alpha=0.5)

        ax.axvline(-1.96, color='gray', linestyle=':', alpha=0.5, label='±1.96')

        ax.set_xlabel('Z-scored Net GC')

        ax.set_ylabel('Count')

        ax.legend(fontsize=7, loc='upper right')

        

        # Add significance annotation

        pval = subj_stats['pval_net'].mean() if 'pval_net' in subj_stats.columns else np.nan

        if not np.isnan(pval):

            sig_str = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'

            ax.text(0.95, 0.95, f'p={pval:.3f} {sig_str}', transform=ax.transAxes, 

                    ha='right', va='top', fontsize=10, fontweight='bold')

    else:

        ax.text(0.5, 0.5, 'No z-scored data', ha='center', va='center', transform=ax.transAxes)



plt.suptitle('Z-scored Granger Causality vs Surrogate Null (Encoding Period)', fontsize=14)

plt.tight_layout()

plt.savefig(f'{plot_dir}/encoding_gc_zscore_summary.pdf', dpi=300, bbox_inches='tight')

plt.show()

## Summary

This notebook computed **multivariate Granger causality (MVGC)** between smoothed HFA signals from different ROIs using statsmodels VAR models.

**Key features:**
- Per-trial F-statistics for bidirectional GC (forward and backward)
- **Surrogate analysis** with 250 trial-shuffled permutations for statistical validation
- Results merged with behavioral data from learn_df (RPE, reward)
- Visualizations of GC distributions, by RPE category, and vs surrogate distributions

**Surrogate method:**
- Trial pairings between source and target ROIs are shuffled (e.g., trial 1 source with trial 5 target)
- This breaks temporal coupling while preserving signal structure within each region
- P-values computed as proportion of surrogates >= real GC

**Files saved:**
- Raw MVGC results: `{savepath}/{roi_pair}_feedback_start_mvgc_raw.csv`
- Merged with behavior: `{savepath}/{roi_pair}_feedback_start_mvgc_merged.csv`
- Figures: `{savepath}/*.pdf`